Import Libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
import pandas as pd
import pickle
from collections import defaultdict


Preprocess of Data

In [3]:
# ============================================================
#  COMPLETE PIPELINE: EXPANDED CANCER COVERAGE
# WITHOUT CLASS WEIGHTS
# ============================================================

import numpy as np
import pandas as pd
import pickle
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder

print("\n" + "="*70)
print("EXPANDED CANCER CLASSIFICATION + ABLATION STUDY")
print("="*70)

# =========================================================
# ABLATION STUDY CONFIGURATION
# =========================================================
USE_RANDOM_LAB = False       
USE_RANDOM_RAD = False       
USE_RANDOM_DRUG_DESC = False  
USE_HIERARCHICAL = True       
RANDOM_SEED = 42

# =========================================================
# PATHS
# =========================================================
EMB_RAD_PATH = r"....cancer_admission_embs_radiology.npy"
EMB_LAB_PATH = r"....solid_cancer_lab_embs.npy"
DRUG_EMB_PATH = r".....cancer_admission_embs_drugs.npy"
DRUG_SEQ_PATH = r"......cancer_drug_sequences.npy"
DDI_PATH = r"......mapped_ddi_pairs.pkl"
DIAGNOSES_PATH = r"....mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"
DRUG2IDX_PATH = r".....drug2idx.pkl"

# =========================================================
# STEP 1: EXPANDED ICD CANCER CODE LISTS
# =========================================================
print("\n" + "-"*40)
print("DEFINING EXPANDED CANCER TYPE MAPPING...")

# Complete ICD-9 Cancer Codes (ALL solid tumors)
EXPANDED_ICD9 = (
    # Head & Neck (140-149)
    "140","141","142","143","144","145","146","147","148","149",
    # Digestive (150-159)
    "150","151","152","153","154","155","156","157","158","159",
    # Respiratory (160-165)
    "160","161","162","163","164","165",
    # Bone/Soft Tissue (170-176)
    "170","171","172","173","174","175","176",
    # Genitourinary (179-189)
    "179","180","181","182","183","184","185","186","187","188","189",
    # Brain/CNS (190-192)
    "190","191","192",
    # Thyroid/Endocrine (193-199)
    "193","194","195","196","197","198","199"
)

# Complete ICD-10 Cancer Codes (ALL solid tumors)
EXPANDED_ICD10 = (
    # Head & Neck (C00-C14)
    "C00","C01","C02","C03","C04","C05","C06","C07","C08","C09","C10","C11","C12","C13","C14",
    # Digestive (C15-C26)
    "C15","C16","C17","C18","C19","C20","C21","C22","C23","C24","C25","C26",
    # Respiratory (C30-C39)
    "C30","C31","C32","C33","C34","C37","C38","C39",
    # Bone/Soft Tissue (C40-C49)
    "C40","C41","C43","C44","C45","C46","C47","C48","C49",
    # Breast (C50)
    "C50",
    # Female Genital (C51-C58)
    "C51","C52","C53","C54","C55","C56","C57","C58",
    # Male Genital (C60-C63)
    "C60","C61","C62","C63",
    # Urinary (C64-C68)
    "C64","C65","C66","C67","C68",
    # Brain/CNS (C69-C72)
    "C69","C70","C71","C72",
    # Thyroid/Endocrine (C73-C75)
    "C73","C74","C75"
)

print(f" Expanded ICD-9 prefixes: {len(EXPANDED_ICD9)}")
print(f" Expanded ICD-10 prefixes: {len(EXPANDED_ICD10)}")

# =========================================================
# STEP 2: COMPREHENSIVE ICD TO CANCER TYPE MAPPING
# =========================================================

# Complete ICD-9 to Cancer Type Mapping
ICD9_CANCER_MAPPING = {
    # Head and Neck (140-149)
    '140': 'HEAD_NECK_CANCER', '141': 'HEAD_NECK_CANCER', '142': 'HEAD_NECK_CANCER',
    '143': 'HEAD_NECK_CANCER', '144': 'HEAD_NECK_CANCER', '145': 'HEAD_NECK_CANCER',
    '146': 'HEAD_NECK_CANCER', '147': 'HEAD_NECK_CANCER', '148': 'HEAD_NECK_CANCER',
    '149': 'HEAD_NECK_CANCER',
    
    # Digestive System
    '150': 'ESOPHAGEAL_CANCER', '151': 'STOMACH_CANCER',
    '152': 'SMALL_INTESTINE_CANCER', '153': 'COLORECTAL_CANCER', '154': 'COLORECTAL_CANCER',
    '155': 'LIVER_CANCER', '156': 'GALLBLADDER_CANCER', '157': 'PANCREATIC_CANCER',
    '158': 'PERITONEAL_CANCER', '159': 'OTHER_DIGESTIVE_CANCER',
    
    # Respiratory
    '160': 'NASAL_CANCER', '161': 'LARYNGEAL_CANCER',
    '162': 'LUNG_CANCER', '163': 'PLEURAL_CANCER', '164': 'THYMUS_CANCER', 
    '165': 'OTHER_RESPIRATORY_CANCER',
    
    # Bone and Soft Tissue
    '170': 'BONE_CANCER', '171': 'SOFT_TISSUE_CANCER', '172': 'MELANOMA',
    '173': 'OTHER_SKIN_CANCER', '174': 'BREAST_CANCER', '175': 'MALE_BREAST_CANCER',
    '176': 'KAPOSI_SARCOMA',
    
    # Genitourinary
    '179': 'UTERINE_CANCER', '180': 'CERVICAL_CANCER', '181': 'PLACENTAL_CANCER',
    '182': 'OVARIAN_CANCER', '183': 'OTHER_FEMALE_GENITAL', '184': 'VULVAR_CANCER',
    '185': 'PROSTATE_CANCER', '186': 'TESTICULAR_CANCER', '187': 'PENILE_CANCER',
    '188': 'BLADDER_CANCER', '189': 'KIDNEY_CANCER',
    
    # Brain and CNS
    '190': 'EYE_CANCER', '191': 'BRAIN_CANCER', '192': 'SPINAL_CORD_CANCER',
    
    # Thyroid and Endocrine
    '193': 'THYROID_CANCER', '194': 'ENDOCRINE_CANCER',
    '195': 'OTHER_CANCER', '196': 'METASTATIC_CANCER', '197': 'SECONDARY_RESPIRATORY',
    '198': 'SECONDARY_DIGESTIVE', '199': 'SECONDARY_CANCER',
}

# Complete ICD-10 to Cancer Type Mapping
ICD10_CANCER_MAPPING = {
    # Head and Neck
    'C00': 'HEAD_NECK_CANCER', 'C01': 'HEAD_NECK_CANCER', 'C02': 'HEAD_NECK_CANCER',
    'C03': 'HEAD_NECK_CANCER', 'C04': 'HEAD_NECK_CANCER', 'C05': 'HEAD_NECK_CANCER',
    'C06': 'HEAD_NECK_CANCER', 'C07': 'HEAD_NECK_CANCER', 'C08': 'HEAD_NECK_CANCER',
    'C09': 'HEAD_NECK_CANCER', 'C10': 'HEAD_NECK_CANCER', 'C11': 'HEAD_NECK_CANCER',
    'C12': 'HEAD_NECK_CANCER', 'C13': 'HEAD_NECK_CANCER', 'C14': 'HEAD_NECK_CANCER',
    
    # Digestive
    'C15': 'ESOPHAGEAL_CANCER', 'C16': 'STOMACH_CANCER', 'C17': 'SMALL_INTESTINE_CANCER',
    'C18': 'COLORECTAL_CANCER', 'C19': 'COLORECTAL_CANCER', 'C20': 'COLORECTAL_CANCER',
    'C21': 'ANAL_CANCER', 'C22': 'LIVER_CANCER', 'C23': 'GALLBLADDER_CANCER',
    'C24': 'BILE_DUCT_CANCER', 'C25': 'PANCREATIC_CANCER', 'C26': 'OTHER_DIGESTIVE_CANCER',
    
    # Respiratory
    'C30': 'NASAL_CANCER', 'C31': 'SINUS_CANCER', 'C32': 'LARYNGEAL_CANCER',
    'C33': 'TRACHEAL_CANCER', 'C34': 'LUNG_CANCER', 'C37': 'THYMUS_CANCER',
    'C38': 'HEART_MEDIASTINAL_CANCER', 'C39': 'OTHER_RESPIRATORY_CANCER',
    
    # Bone and Soft Tissue
    'C40': 'BONE_CANCER', 'C41': 'BONE_CANCER', 'C43': 'MELANOMA',
    'C44': 'OTHER_SKIN_CANCER', 'C45': 'MESOTHELIOMA', 'C46': 'KAPOSI_SARCOMA',
    'C47': 'PERIPHERAL_NERVE_CANCER', 'C48': 'RETROPERITONEAL_CANCER', 'C49': 'SOFT_TISSUE_CANCER',
    
    # Breast
    'C50': 'BREAST_CANCER',
    
    # Female Genital
    'C51': 'VULVAR_CANCER', 'C52': 'VAGINAL_CANCER', 'C53': 'CERVICAL_CANCER',
    'C54': 'ENDOMETRIAL_CANCER', 'C55': 'UTERINE_CANCER', 'C56': 'OVARIAN_CANCER',
    'C57': 'OTHER_FEMALE_GENITAL', 'C58': 'PLACENTAL_CANCER',
    
    # Male Genital
    'C60': 'PENILE_CANCER', 'C61': 'PROSTATE_CANCER', 'C62': 'TESTICULAR_CANCER',
    'C63': 'OTHER_MALE_GENITAL',
    
    # Urinary
    'C64': 'KIDNEY_CANCER', 'C65': 'RENAL_PELVIS_CANCER', 'C66': 'URETERAL_CANCER',
    'C67': 'BLADDER_CANCER', 'C68': 'OTHER_URINARY_CANCER',
    
    # Brain and CNS
    'C69': 'EYE_CANCER', 'C70': 'MENINGEAL_CANCER', 'C71': 'BRAIN_CANCER',
    'C72': 'SPINAL_CORD_CANCER',
    
    # Thyroid and Endocrine
    'C73': 'THYROID_CANCER', 'C74': 'ADRENAL_CANCER', 'C75': 'OTHER_ENDOCRINE_CANCER',
}

print(f"✅ ICD-9 mapped: {len(ICD9_CANCER_MAPPING)} codes")
print(f"✅ ICD-10 mapped: {len(ICD10_CANCER_MAPPING)} codes")

# =========================================================
# STEP 3: FUNCTION TO MAP ICD TO CANCER TYPE
# =========================================================

def map_icd_to_cancer_type(icd_code, icd_version):
    """Map ICD code to specific cancer type using comprehensive mappings"""
    icd_code = str(icd_code).upper().strip()
    
    if icd_version == 9:
        for code_prefix, cancer_type in ICD9_CANCER_MAPPING.items():
            if icd_code.startswith(code_prefix):
                return cancer_type
    elif icd_version == 10:
        for code_prefix, cancer_type in ICD10_CANCER_MAPPING.items():
            if icd_code.startswith(code_prefix):
                return cancer_type
    
    return 'OTHER_CANCER'

# =========================================================
# STEP 4: HIERARCHICAL COARSE CLASS MAPPING
# =========================================================

# Group similar cancer types for balanced classification
COARSE_CANCER_GROUPS = {
    'LUNG_CANCER': ['LUNG_CANCER', 'TRACHEAL_CANCER'],
    'BREAST_CANCER': ['BREAST_CANCER', 'MALE_BREAST_CANCER'],
    'COLORECTAL_CANCER': ['COLORECTAL_CANCER', 'ANAL_CANCER'],
    'PROSTATE_CANCER': ['PROSTATE_CANCER'],
    'BLADDER_CANCER': ['BLADDER_CANCER'],
    'KIDNEY_CANCER': ['KIDNEY_CANCER', 'RENAL_PELVIS_CANCER'],
    'STOMACH_CANCER': ['STOMACH_CANCER'],
    'LIVER_CANCER': ['LIVER_CANCER', 'BILE_DUCT_CANCER'],
    'PANCREATIC_CANCER': ['PANCREATIC_CANCER'],
    'ESOPHAGEAL_CANCER': ['ESOPHAGEAL_CANCER'],
    'OVARIAN_CANCER': ['OVARIAN_CANCER'],
    'CERVICAL_CANCER': ['CERVICAL_CANCER'],
    'UTERINE_CANCER': ['UTERINE_CANCER', 'ENDOMETRIAL_CANCER'],
    'HEAD_NECK_CANCER': ['HEAD_NECK_CANCER', 'LARYNGEAL_CANCER', 'NASAL_CANCER', 
                         'SINUS_CANCER', 'ORAL_CANCER', 'SALIVARY_CANCER'],
    'THYROID_CANCER': ['THYROID_CANCER'],
    'BRAIN_CANCER': ['BRAIN_CANCER', 'SPINAL_CORD_CANCER', 'MENINGEAL_CANCER'],
    'MELANOMA': ['MELANOMA', 'OTHER_SKIN_CANCER'],
    'KAPOSI_SARCOMA': ['KAPOSI_SARCOMA', 'MESOTHELIOMA'],
    'OTHER_CANCER': ['OTHER_CANCER', 'METASTATIC_CANCER', 'SECONDARY_CANCER',
                     'OTHER_DIGESTIVE_CANCER', 'OTHER_RESPIRATORY_CANCER',
                     'OTHER_FEMALE_GENITAL', 'OTHER_MALE_GENITAL', 'OTHER_URINARY_CANCER',
                     'OTHER_ENDOCRINE_CANCER', 'PERITONEAL_CANCER', 'PLEURAL_CANCER',
                     'THYMUS_CANCER', 'HEART_MEDIASTINAL_CANCER', 'RETROPERITONEAL_CANCER',
                     'PERIPHERAL_NERVE_CANCER', 'SOFT_TISSUE_CANCER', 'BONE_CANCER',
                     'EYE_CANCER', 'ADRENAL_CANCER', 'TESTICULAR_CANCER', 'PENILE_CANCER',
                     'VULVAR_CANCER', 'VAGINAL_CANCER', 'PLACENTAL_CANCER', 'GALLBLADDER_CANCER',
                     'SMALL_INTESTINE_CANCER', 'URETERAL_CANCER']
}

def map_to_coarse_class(cancer_type):
    """Map fine-grained cancer type to coarse category"""
    for coarse_group, fine_types in COARSE_CANCER_GROUPS.items():
        if cancer_type in fine_types:
            return coarse_group
    return 'OTHER_CANCER'

# =========================================================
# STEP 5: HELPER FUNCTION FOR RANDOM EMBEDDINGS
# =========================================================
def maybe_replace_with_random(embeddings, use_random, modality_name, seed=None):
    if not use_random:
        print(f"✓ Using REAL {modality_name} embeddings")
        return embeddings
    
    print(f" Using RANDOM {modality_name} embeddings (seed={seed})")
    if seed is not None:
        np.random.seed(seed)
    
    mean = np.mean(embeddings)
    std = np.std(embeddings)
    random_embs = np.random.normal(mean, std, size=embeddings.shape)
    
    original_norms = np.linalg.norm(embeddings, axis=1)
    random_norms = np.linalg.norm(random_embs, axis=1)
    scale_factors = original_norms / (random_norms + 1e-8)
    random_embs = random_embs * scale_factors[:, np.newaxis]
    
    return random_embs.astype(embeddings.dtype)

# =========================================================
# STEP 6: LOAD AND PROCESS EMBEDDINGS
# =========================================================
print("\n" + "="*60)
print("LOADING AND PROCESSING EMBEDDINGS")
print("="*60)

X_rad = np.load(EMB_RAD_PATH)
hadm_ids_rad = np.load(EMB_RAD_PATH.replace(".npy","_hadm_ids.npy"))
subject_ids_rad = np.load(EMB_RAD_PATH.replace(".npy","_subject_ids.npy"))

X_lab = np.load(EMB_LAB_PATH)
hadm_ids_lab = np.load(EMB_LAB_PATH.replace(".npy","_hadm_ids.npy"))
subject_ids_lab = np.load(EMB_LAB_PATH.replace(".npy","_subject_ids.npy"))

print(f"\nOriginal shapes:")
print(f"  Radiology: {X_rad.shape}")
print(f"  Lab: {X_lab.shape}")

# Apply random embeddings
X_lab = maybe_replace_with_random(X_lab, USE_RANDOM_LAB, "LAB", RANDOM_SEED)
X_rad = maybe_replace_with_random(X_rad, USE_RANDOM_RAD, "RADIOLOGY", RANDOM_SEED)

# Align admissions
common_hadm_ids = np.intersect1d(hadm_ids_rad, hadm_ids_lab)
idx_rad = np.isin(hadm_ids_rad, common_hadm_ids)
idx_lab = np.isin(hadm_ids_lab, common_hadm_ids)

X_rad = X_rad[idx_rad]
X_lab = X_lab[idx_lab]
hadm_ids = hadm_ids_lab[idx_lab]
subject_ids = subject_ids_lab[idx_lab]

print(f"\nAfter alignment:")
print(f"  Radiology: {X_rad.shape}")
print(f"  Lab: {X_lab.shape}")

# =========================================================
# STEP 7: LOAD DRUG DATA
# =========================================================
print("\n" + "-"*40)
print("LOADING DRUG DATA...")

X_drug = np.load(DRUG_EMB_PATH)
hadm_ids_drug = np.load(DRUG_EMB_PATH.replace(".npy","_hadm_ids.npy"))
drug_sequences = np.load(DRUG_SEQ_PATH)
drug_lengths = np.load(DRUG_SEQ_PATH.replace(".npy","_lengths.npy"))

print(f"Drug embeddings shape: {X_drug.shape}")
print(f"Drug sequences shape: {drug_sequences.shape}")

# Apply random embeddings to drug descriptions
X_drug = maybe_replace_with_random(X_drug, USE_RANDOM_DRUG_DESC, "DRUG DESCRIPTION", RANDOM_SEED)

# =========================================================
# STEP 8: ALIGN ALL DATA
# =========================================================
print("\n" + "-"*40)
print("ALIGNING DATA...")

drug_idx = {hid: i for i, hid in enumerate(hadm_ids_drug)}

aligned_X_drug = []
aligned_drug_sequences = []
aligned_drug_lengths = []
aligned_X_lab = []
aligned_X_rad = []
aligned_subject_ids = []
aligned_hadm_ids = []
missing_drug = 0

for i, hid in enumerate(hadm_ids):
    if hid in drug_idx:
        j = drug_idx[hid]
        aligned_X_drug.append(X_drug[j])
        aligned_drug_sequences.append(drug_sequences[j])
        aligned_drug_lengths.append(drug_lengths[j])
        aligned_X_lab.append(X_lab[i])
        aligned_X_rad.append(X_rad[i])
        aligned_subject_ids.append(subject_ids[i])
        aligned_hadm_ids.append(hid)
    else:
        missing_drug += 1

print(f"Admissions missing drug data: {missing_drug}")

# Stack arrays
X_drug = np.stack(aligned_X_drug)
drug_sequences = np.stack(aligned_drug_sequences)
drug_lengths = np.array(aligned_drug_lengths)
X_lab = np.stack(aligned_X_lab)
X_rad = np.stack(aligned_X_rad)
subject_ids = np.array(aligned_subject_ids)
hadm_ids = np.array(aligned_hadm_ids)

print(f"\nAligned shapes:")
print(f"  Lab: {X_lab.shape}")
print(f"  Rad: {X_rad.shape}")
print(f"  Drug desc: {X_drug.shape}")
print(f"  Drug seq: {drug_sequences.shape}")

# =========================================================
# STEP 9: LOAD AND MAP DIAGNOSIS LABELS (UPDATED)
# =========================================================
print("\n" + "-"*40)
print("LOADING DIAGNOSIS LABELS...")

diag = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=["hadm_id","icd_code","icd_version","seq_num"]
)
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()

# Use EXPANDED cancer filters
mask = (
    ((diag.icd_version==9) & diag.icd_code.str.startswith(EXPANDED_ICD9)) |
    ((diag.icd_version==10) & diag.icd_code.str.startswith(EXPANDED_ICD10))
)

hf_diag = diag[mask]
hadm_to_cancer_type = {}

for hid, g in hf_diag.groupby("hadm_id"):
    primary = g[g.seq_num.isin([1,2])]
    if len(primary) > 0:
        row = primary.sort_values("seq_num").iloc[0]
        cancer_type = map_icd_to_cancer_type(row["icd_code"], row["icd_version"])
        hadm_to_cancer_type[hid] = cancer_type
    else:
        # If no primary, take most common cancer type in admission
        most_common = g['icd_code'].mode()
        if len(most_common) > 0:
            cancer_type = map_icd_to_cancer_type(most_common[0], g.iloc[0]['icd_version'])
            hadm_to_cancer_type[hid] = cancer_type
        else:
            hadm_to_cancer_type[hid] = "OTHER_CANCER"

# Apply diagnosis filter
final_mask = np.array([hid in hadm_to_cancer_type for hid in hadm_ids])
X_lab = X_lab[final_mask]
X_rad = X_rad[final_mask]
X_drug = X_drug[final_mask]
drug_sequences = drug_sequences[final_mask]
drug_lengths = drug_lengths[final_mask]
subject_ids = subject_ids[final_mask]
hadm_ids = hadm_ids[final_mask]

y_hf_fine = np.array([hadm_to_cancer_type[hid] for hid in hadm_ids])

print(f"After diagnosis filter: {len(y_hf_fine)} admissions")

# =========================================================
# STEP 10: COLLAPSE VERY RARE CLASSES
# =========================================================
print("\n" + "-"*40)
print("COLLAPSING VERY RARE CLASSES...")

MIN_SAMPLES = 100
counts = pd.Series(y_hf_fine).value_counts()
rare = counts[counts < MIN_SAMPLES].index

y_hf_fine = np.array([
    "RARE_CANCER" if lbl in rare else lbl
    for lbl in y_hf_fine
])

print("\nFine-grained label distribution:")
label_counts = pd.Series(y_hf_fine).value_counts()
for label, count in label_counts.items():
    pct = count / len(y_hf_fine) * 100
    bar = '' * int(pct / 2)
    print(f"  {label:30s}: {count:5d} ({pct:5.1f}%) {bar}")

# =========================================================
# STEP 11: APPLY HIERARCHICAL MAPPING
# =========================================================
print("\n" + "-"*40)
print("APPLYING HIERARCHICAL MAPPING...")

if USE_HIERARCHICAL:
    y_hf = np.array([map_to_coarse_class(lbl) for lbl in y_hf_fine])
    print("\n Using HIERARCHICAL coarse classes:")
else:
    y_hf = y_hf_fine.copy()
    print("\n Using FINE-grained original classes:")

label_counts = pd.Series(y_hf).value_counts()
total = len(y_hf)
print(f"\nFinal label distribution ({len(label_counts)} classes):")
for label, count in label_counts.items():
    pct = count / total * 100
    bar = '' * int(pct / 2)
    print(f"  {label:30s}: {count:5d} ({pct:5.1f}%) {bar}")

# =========================================================
# STEP 12: AGGREGATE BY PATIENT
# =========================================================
print("\n" + "-"*40)
print("AGGREGATING BY PATIENT...")

patient_to_lab = defaultdict(list)
patient_to_rad = defaultdict(list)
patient_to_drug_desc = defaultdict(list)
patient_to_labels = defaultdict(list)
patient_to_drugs = defaultdict(list)

for lab, rad, drug_desc, lbl, drug_seq, pid in zip(
    X_lab, X_rad, X_drug, y_hf, drug_sequences, subject_ids
):
    patient_to_lab[pid].append(lab)
    patient_to_rad[pid].append(rad)
    patient_to_drug_desc[pid].append(drug_desc)
    patient_to_labels[pid].append(lbl)
    patient_to_drugs[pid].append(drug_seq)

lab_sequences_by_patient = [np.stack(v) for v in patient_to_lab.values()]
rad_sequences_by_patient = [np.stack(v) for v in patient_to_rad.values()]
drug_desc_sequences_by_patient = [np.stack(v) for v in patient_to_drug_desc.values()]
drug_labels_by_patient = [np.stack(v) for v in patient_to_drugs.values()]
labels_by_patient = [np.array(v) for v in patient_to_labels.values()]
patient_ids = list(patient_to_lab.keys())

print(f"\nFinal dataset:")
print(f"  Patients: {len(patient_ids)}")
print(f"  Total admissions: {sum(len(seq) for seq in labels_by_patient)}")

# =========================================================
# STEP 13: LOAD DDI MATRIX
# =========================================================
print("\n" + "-"*40)
print("LOADING DDI MATRIX...")

with open(DDI_PATH, "rb") as f:
    mapped_ddi_pairs = pickle.load(f)

with open(DRUG2IDX_PATH, "rb") as f:
    drug2idx = pickle.load(f)

def normalize_drug_name(name):
    if name is None:
        return ""
    name = name.lower().strip()
    if name.startswith("*nf*"):
        name = name[4:].strip()
    return name

norm_drug2idx = {normalize_drug_name(d): idx for d, idx in drug2idx.items()}
n_drugs = len(norm_drug2idx)
print(f"Number of drugs: {n_drugs}")

severity_weight = {"minor": 0.5, "moderate": 2.0, "major": 5.0}

def build_ddi_severity_matrices(ddi_pairs, drug2idx, n_drugs, severity_weight):
    ddi_matrix = np.zeros((n_drugs, n_drugs), dtype=np.float32)
    for drug1, drug2, severity in ddi_pairs:
        if drug1 in drug2idx and drug2 in drug2idx:
            i, j = drug2idx[drug1], drug2idx[drug2]
            weight = severity_weight.get(severity, 1.0)
            ddi_matrix[i, j] = weight
            ddi_matrix[j, i] = weight
    return ddi_matrix

ddi_severity_matrix = build_ddi_severity_matrices(
    ddi_pairs=mapped_ddi_pairs,
    drug2idx=norm_drug2idx,
    n_drugs=n_drugs,
    severity_weight=severity_weight
)

print(f"DDI severity matrix shape: {ddi_severity_matrix.shape}")

# =========================================================
# STEP 14: CREATE LABEL ENCODER FOR TRAINING
# =========================================================
print("\n" + "-"*40)
print("CREATING LABEL ENCODER...")

le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(seq) for seq in labels_by_patient]
n_classes = len(le_hf.classes_)

print(f"Number of classes for training: {n_classes}")
print(f"Classes: {list(le_hf.classes_)}")

# =========================================================
# STEP 15: FINAL VALIDATION
# =========================================================
print("\n" + "="*60)
print("VALIDATION")
print("="*60)

n = len(hadm_ids)
assert X_lab.shape[0] == n, f"Lab shape mismatch"
assert X_rad.shape[0] == n, f"Rad shape mismatch"
assert X_drug.shape[0] == n, f"Drug desc shape mismatch"

print("\n PREPROCESSING COMPLETE!")
print(f"\n Ablation configuration:")
print(f"   - LAB: {'RANDOM' if USE_RANDOM_LAB else 'REAL'}")
print(f"   - RADIOLOGY: {'RANDOM' if USE_RANDOM_RAD else 'REAL'}")
print(f"   - DRUG DESCRIPTION: {'RANDOM' if USE_RANDOM_DRUG_DESC else 'REAL'}")
print(f"   - HIERARCHICAL CLASSES: {'YES' if USE_HIERARCHICAL else 'NO'}")

print(f"\n Dataset statistics:")
print(f"   - Patients: {len(patient_ids)}")
print(f"   - Admissions: {n}")
print(f"   - Classes: {n_classes}")
print(f"   - Drugs: {n_drugs}")

print(f"\n Final class distribution:")
for cls, count in label_counts.items():
    pct = count / total * 100
    bar = '' * int(pct / 2)
    print(f"  {cls:30s}: {count:5d} ({pct:5.1f}%) {bar}")


EXPANDED CANCER CLASSIFICATION + ABLATION STUDY

----------------------------------------
DEFINING EXPANDED CANCER TYPE MAPPING...
✅ Expanded ICD-9 prefixes: 54
✅ Expanded ICD-10 prefixes: 69
✅ ICD-9 mapped: 54 codes
✅ ICD-10 mapped: 69 codes

LOADING AND PROCESSING EMBEDDINGS

Original shapes:
  Radiology: (12708, 2560)
  Lab: (19866, 2560)
✓ Using REAL LAB embeddings
✓ Using REAL RADIOLOGY embeddings

After alignment:
  Radiology: (12093, 2560)
  Lab: (12093, 2560)

----------------------------------------
LOADING DRUG DATA...
Drug embeddings shape: (21158, 2560)
Drug sequences shape: (21173, 689)
✓ Using REAL DRUG DESCRIPTION embeddings

----------------------------------------
ALIGNING DATA...
Admissions missing drug data: 41

Aligned shapes:
  Lab: (12052, 2560)
  Rad: (12052, 2560)
  Drug desc: (12052, 2560)
  Drug seq: (12052, 689)

----------------------------------------
LOADING DIAGNOSIS LABELS...
After diagnosis filter: 12052 admissions

------------------------------------

Random Forest Classifier

In [4]:
# ============================================================
#  CREATE DRUG LABELS ENCODING (Matching GRU)
# ============================================================
print("\n" + "="*60)
print("CREATING DRUG LABELS ENCODING")
print("="*60)

# Get number of drugs from DDI matrix
n_drugs = ddi_severity_matrix.shape[0]
print(f"Number of drugs: {n_drugs}")

# Convert drug sequences to multi-hot encoded vectors
drug_labels_by_patient_enc = []

for patient_drug_seq in drug_labels_by_patient:
    # Initialize multi-hot matrix for this patient
    seq_enc = np.zeros((len(patient_drug_seq), n_drugs), dtype=np.float32)
    
    for t, drugs in enumerate(patient_drug_seq):
        if len(drugs) > 0:
            # Ensure drug indices are valid
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    
    drug_labels_by_patient_enc.append(seq_enc)

print(f"Created drug history for {len(drug_labels_by_patient_enc)} patients")
print(f"Example shape: {drug_labels_by_patient_enc[0].shape}")

# ============================================================
#  ENCODE LABELS FOR RANDOM FOREST
# ============================================================
from sklearn.preprocessing import LabelEncoder

# Encode cancer labels
label_encoder = LabelEncoder()
all_labels = []

# Flatten all patient label sequences to fit encoder
for lbl_seq in labels_by_patient:
    all_labels.extend(lbl_seq)

# Fit encoder on all unique labels
label_encoder.fit(all_labels)

# Transform each patient's labels
labels_by_patient_enc = []
for lbl_seq in labels_by_patient:
    labels_by_patient_enc.append(label_encoder.transform(lbl_seq))

# Get number of classes
n_classes = len(label_encoder.classes_)
print(f"\nNumber of cancer classes: {n_classes}")
print(f"Classes: {label_encoder.classes_}")

# ============================================================
# FAITHFUL RANDOM FOREST - MATCHES GRU EXACTLY
# ============================================================

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Settings (match GRU)
# -----------------------------
n_estimators = 300
max_depth = 20
random_state = 42

# -----------------------------
# FIXED: Use ALL 4 modalities (match GRU)
# -----------------------------
def extract_features_with_drug_history(lab_seqs, rad_seqs, drug_desc_seqs, drug_history_seqs):
    """
    Extract features including drug history.
    Matches GRU's input exactly.
    """
    features = []
    valid_indices = []
    
    for i, (lab, rad, drug_desc, drug_history) in enumerate(zip(
        lab_seqs, rad_seqs, drug_desc_seqs, drug_history_seqs
    )):
        # Need at least 2 admissions to predict next (same as GRU)
        if len(lab) < 2:
            continue
            
        # Use features from admission t to predict t+1
        # GRU at timestep t uses: lab[t], rad[t], drug_desc[t], drug_history[t]
        x = np.concatenate([
            lab[-2],           # Lab at t (second-to-last)
            rad[-2],           # Radiology at t
            drug_desc[-2],     # Drug description at t
            drug_history[-2]   # Drug history at t (MISSING BEFORE!)
        ])
        
        features.append(x)
        valid_indices.append(i)
    
    if len(features) == 0:
        return np.array([]), np.array([])
    
    return np.array(features), np.array(valid_indices)

# -----------------------------
# Prepare drug history as features (multi-hot encoded)
# -----------------------------
# Use drug_labels_by_patient_enc created above
drug_history_sequences = []
for i in range(len(drug_labels_by_patient_enc)):
    drug_history_sequences.append(drug_labels_by_patient_enc[i])

print(f"\nDrug history sequences prepared: {len(drug_history_sequences)} patients")

# -----------------------------
# CV Setup (match GRU exactly)
# -----------------------------
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

micro_f1_scores = []
macro_f1_scores = []
roc_auc_scores = []

print("\n" + "="*60)
print("FAITHFUL RF - MATCHING GRU EXACTLY")
print("="*60)
print("✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)")
print("✓ Same patient splitting strategy")
print("✓ Same prediction task (next admission cancer type)")
print("="*60)

# ============================================================
# CROSS-VALIDATION LOOP
# ============================================================
for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    
    print(f"\n{'='*60}")
    print(f"Fold {fold}/10")
    print(f"{'='*60}")
    
    # Train/val split (match GRU)
    train_idx, val_idx = train_test_split(
        train_idx, 
        test_size=0.2, 
        random_state=42, 
        stratify=[strat_labels[i] for i in train_idx]
    )
    
    # Build TRAIN set with ALL modalities
    train_lab = [lab_sequences_by_patient[i] for i in train_idx]
    train_rad = [rad_sequences_by_patient[i] for i in train_idx]
    train_drug_desc = [drug_desc_sequences_by_patient[i] for i in train_idx]
    train_drug_history = [drug_history_sequences[i] for i in train_idx]
    train_labels = [labels_by_patient_enc[i] for i in train_idx]
    
    X_train_raw, train_valid_idx = extract_features_with_drug_history(
        train_lab, train_rad, train_drug_desc, train_drug_history
    )
    y_train = []
    for idx in train_valid_idx:
        seq = train_labels[idx]
        if len(seq) >= 2:
            y_train.append(seq[-1])
    y_train = np.array(y_train)
    
    if len(X_train_raw) == 0:
        print(f" Warning: No valid training samples in fold {fold}")
        continue
    
    # Build TEST set with ALL modalities
    test_lab = [lab_sequences_by_patient[i] for i in test_idx]
    test_rad = [rad_sequences_by_patient[i] for i in test_idx]
    test_drug_desc = [drug_desc_sequences_by_patient[i] for i in test_idx]
    test_drug_history = [drug_history_sequences[i] for i in test_idx]
    test_labels = [labels_by_patient_enc[i] for i in test_idx]
    
    X_test_raw, test_valid_idx = extract_features_with_drug_history(
        test_lab, test_rad, test_drug_desc, test_drug_history
    )
    y_test = []
    for idx in test_valid_idx:
        seq = test_labels[idx]
        if len(seq) >= 2:
            y_test.append(seq[-1])
    y_test = np.array(y_test)
    
    if len(X_test_raw) == 0:
        print(f" Warning: No valid test samples in fold {fold}")
        continue
    
    print(f"Train samples: {X_train_raw.shape[0]}, Test samples: {X_test_raw.shape[0]}")
    print(f"Feature dimension: {X_train_raw.shape[1]}")
    print(f"Classes in train: {np.unique(y_train)}")
    print(f"Classes in test: {np.unique(y_test)}")
    
    # Scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)
    
    # Train RF with class balancing
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=random_state,
        class_weight="balanced"
    )
    
    rf.fit(X_train, y_train)
    
    # Predict
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)
    
    # Metrics
    micro_f1 = f1_score(y_test, y_pred, average="micro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    
    try:
        roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
    except Exception as e:
        print(f"ROC-AUC computation failed: {e}")
        roc_auc = 0.0
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    roc_auc_scores.append(roc_auc)
    
    print(f"\nFold {fold} Results:")
    print(f"  Micro-F1 : {micro_f1:.4f}")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")

# ============================================================
# 📊 FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FAITHFUL RANDOM FOREST - 10-FOLD CV RESULTS")
print("="*60)

if len(micro_f1_scores) > 0:
    print(f"Micro-F1 : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC  : {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")
    
    print("\nPer-fold results:")
    for i, (mf, maf, ra) in enumerate(zip(micro_f1_scores, macro_f1_scores, roc_auc_scores), 1):
        print(f"  Fold {i:2d}: Micro-F1={mf:.4f}, Macro-F1={maf:.4f}, ROC-AUC={ra:.4f}")
else:
    print("❌ No valid folds completed!")


CREATING DRUG LABELS ENCODING
Number of drugs: 2374
Created drug history for 7011 patients
Example shape: (1, 2374)

Number of cancer classes: 7
Classes: ['BLADDER_CANCER' 'BREAST_CANCER' 'COLORECTAL_CANCER' 'HEAD_NECK_CANCER'
 'LUNG_CANCER' 'OTHER_CANCER' 'PROSTATE_CANCER']

Drug history sequences prepared: 7011 patients

FAITHFUL RF - MATCHING GRU EXACTLY
✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)
✓ Same patient splitting strategy
✓ Same prediction task (next admission cancer type)

Fold 1/10
Train samples: 1867, Test samples: 242
Feature dimension: 10054
Classes in train: [0 1 2 3 4 5 6]
Classes in test: [0 1 2 3 4 5 6]

Fold 1 Results:
  Micro-F1 : 0.5331
  Macro-F1 : 0.3038
  ROC-AUC  : 0.7910

Fold 2/10
Train samples: 1834, Test samples: 271
Feature dimension: 10054
Classes in train: [0 1 2 3 4 5 6]
Classes in test: [0 1 2 3 4 5 6]

Fold 2 Results:
  Micro-F1 : 0.5314
  Macro-F1 : 0.3040
  ROC-AUC  : 0.8076

Fold 3/10
Train samples: 1825, Test samples: 259
F

XGBoost Classifier

In [19]:
# ============================================================
#  CREATE DRUG LABELS ENCODING (Matching GRU)
# ============================================================
print("\n" + "="*60)
print("CREATING DRUG LABELS ENCODING")
print("="*60)

# Get number of drugs from DDI matrix
n_drugs = ddi_severity_matrix.shape[0]
print(f"Number of drugs: {n_drugs}")

# Convert drug sequences to multi-hot encoded vectors
drug_labels_by_patient_enc = []

for patient_drug_seq in drug_labels_by_patient:
    # Initialize multi-hot matrix for this patient
    seq_enc = np.zeros((len(patient_drug_seq), n_drugs), dtype=np.float32)
    
    for t, drugs in enumerate(patient_drug_seq):
        if len(drugs) > 0:
            # Ensure drug indices are valid
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    
    drug_labels_by_patient_enc.append(seq_enc)

print(f"Created drug history for {len(drug_labels_by_patient_enc)} patients")
print(f"Example shape: {drug_labels_by_patient_enc[0].shape}")

# ============================================================
#  ENCODE LABELS FOR XGBOOST
# ============================================================
from sklearn.preprocessing import LabelEncoder

# Encode cancer labels
label_encoder = LabelEncoder()
all_labels = []

# Flatten all patient label sequences to fit encoder
for lbl_seq in labels_by_patient:
    all_labels.extend(lbl_seq)

# Fit encoder on all unique labels
label_encoder.fit(all_labels)

# Transform each patient's labels
labels_by_patient_enc = []
for lbl_seq in labels_by_patient:
    labels_by_patient_enc.append(label_encoder.transform(lbl_seq))

# Get number of classes
n_classes = len(label_encoder.classes_)
print(f"\nNumber of cancer classes: {n_classes}")
print(f"Classes: {label_encoder.classes_}")

# ============================================================
#  FAITHFUL XGBOOST - MATCHES GRU EXACTLY
# ============================================================

import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Settings (optimized for XGBoost)
# -----------------------------
xgb_params = {
    'n_estimators': 300,
    'max_depth': 6,           # XGBoost typically uses smaller depth
    'learning_rate': 0.05,    # Lower learning rate for better generalization
    'subsample': 0.8,         # Row sampling
    'colsample_bytree': 0.8,  # Column sampling
    'min_child_weight': 3,    # Minimum instances per leaf
    'gamma': 0.1,             # Minimum loss reduction for split
    'reg_alpha': 0.1,         # L1 regularization
    'reg_lambda': 1.0,        # L2 regularization
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

print("\n" + "="*60)
print("FAITHFUL XGBOOST - MATCHING GRU EXACTLY")
print("="*60)
print("✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)")
print("✓ Same patient splitting strategy")
print("✓ Same prediction task (next admission cancer type)")
print("✓ XGBoost parameters:")
for key, value in xgb_params.items():
    print(f"    {key}: {value}")
print("="*60)

# -----------------------------
# Use ALL 4 modalities (match GRU)
# -----------------------------
def extract_features_with_drug_history(lab_seqs, rad_seqs, drug_desc_seqs, drug_history_seqs):
    """
    Extract features including drug history.
    Matches GRU's input exactly.
    """
    features = []
    valid_indices = []
    
    for i, (lab, rad, drug_desc, drug_history) in enumerate(zip(
        lab_seqs, rad_seqs, drug_desc_seqs, drug_history_seqs
    )):
        # Need at least 2 admissions to predict next (same as GRU)
        if len(lab) < 2:
            continue
            
        # Use features from admission t to predict t+1
        x = np.concatenate([
            lab[-2],           # Lab at t
            rad[-2],           # Radiology at t
            drug_desc[-2],     # Drug description at t
            drug_history[-2]   # Drug history at t
        ])
        
        features.append(x)
        valid_indices.append(i)
    
    if len(features) == 0:
        return np.array([]), np.array([])
    
    return np.array(features), np.array(valid_indices)

# -----------------------------
# Prepare drug history as features (multi-hot encoded)
# -----------------------------
drug_history_sequences = []
for i in range(len(drug_labels_by_patient_enc)):
    drug_history_sequences.append(drug_labels_by_patient_enc[i])

print(f"\nDrug history sequences prepared: {len(drug_history_sequences)} patients")

# -----------------------------
# CV Setup (match GRU exactly)
# -----------------------------
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
micro_f1_scores = []
macro_f1_scores = []
roc_auc_scores = []

# ============================================================
#  CROSS-VALIDATION LOOP
# ============================================================
for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    
    print(f"\n{'='*60}")
    print(f"XGBoost Fold {fold}/10")
    print(f"{'='*60}")
    
    # Train/val split (match GRU)
    train_idx, val_idx = train_test_split(
        train_idx, 
        test_size=0.2, 
        random_state=42, 
        stratify=[strat_labels[i] for i in train_idx]
    )
    
    # Build TRAIN set with ALL modalities
    train_lab = [lab_sequences_by_patient[i] for i in train_idx]
    train_rad = [rad_sequences_by_patient[i] for i in train_idx]
    train_drug_desc = [drug_desc_sequences_by_patient[i] for i in train_idx]
    train_drug_history = [drug_history_sequences[i] for i in train_idx]
    train_labels = [labels_by_patient_enc[i] for i in train_idx]
    
    X_train_raw, train_valid_idx = extract_features_with_drug_history(
        train_lab, train_rad, train_drug_desc, train_drug_history
    )
    y_train = []
    for idx in train_valid_idx:
        seq = train_labels[idx]
        if len(seq) >= 2:
            y_train.append(seq[-1])
    y_train = np.array(y_train)
    
    if len(X_train_raw) == 0:
        print(f" Warning: No valid training samples in fold {fold}")
        continue
    
    # Build VALIDATION set for early stopping
    val_lab = [lab_sequences_by_patient[i] for i in val_idx]
    val_rad = [rad_sequences_by_patient[i] for i in val_idx]
    val_drug_desc = [drug_desc_sequences_by_patient[i] for i in val_idx]
    val_drug_history = [drug_history_sequences[i] for i in val_idx]
    val_labels = [labels_by_patient_enc[i] for i in val_idx]
    
    X_val_raw, val_valid_idx = extract_features_with_drug_history(
        val_lab, val_rad, val_drug_desc, val_drug_history
    )
    y_val = []
    for idx in val_valid_idx:
        seq = val_labels[idx]
        if len(seq) >= 2:
            y_val.append(seq[-1])
    y_val = np.array(y_val)
    
    # Build TEST set with ALL modalities
    test_lab = [lab_sequences_by_patient[i] for i in test_idx]
    test_rad = [rad_sequences_by_patient[i] for i in test_idx]
    test_drug_desc = [drug_desc_sequences_by_patient[i] for i in test_idx]
    test_drug_history = [drug_history_sequences[i] for i in test_idx]
    test_labels = [labels_by_patient_enc[i] for i in test_idx]
    
    X_test_raw, test_valid_idx = extract_features_with_drug_history(
        test_lab, test_rad, test_drug_desc, test_drug_history
    )
    y_test = []
    for idx in test_valid_idx:
        seq = test_labels[idx]
        if len(seq) >= 2:
            y_test.append(seq[-1])
    y_test = np.array(y_test)
    
    if len(X_test_raw) == 0:
        print(f" Warning: No valid test samples in fold {fold}")
        continue
    
    print(f"Train samples: {X_train_raw.shape[0]}")
    print(f"Val samples: {X_val_raw.shape[0]}")
    print(f"Test samples: {X_test_raw.shape[0]}")
    print(f"Feature dimension: {X_train_raw.shape[1]}")
    print(f"Classes in train: {np.unique(y_train)}")
    
    # Scale features (XGBoost doesn't strictly need scaling, but helps with convergence)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)
    
    # Update n_estimators for early stopping
    params = xgb_params.copy()
    params['num_class'] = n_classes
    
    # Create DMatrix objects for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test)
    
    # Train with early stopping
    evals = [(dtrain, 'train'), (dval, 'eval')]
    
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,  # More rounds, early stopping will stop early
        evals=evals,
        early_stopping_rounds=30,
        verbose_eval=False
    )
    
    # Predict
    y_proba = model.predict(dtest)
    y_pred = np.argmax(y_proba, axis=1)
    
    # Metrics
    micro_f1 = f1_score(y_test, y_pred, average="micro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    
    try:
        roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
    except Exception as e:
        print(f"ROC-AUC computation failed: {e}")
        roc_auc = 0.0
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    roc_auc_scores.append(roc_auc)
    
    print(f"\nFold {fold} Results:")
    print(f"  Micro-F1 : {micro_f1:.4f}")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  Best iteration: {model.best_iteration}")

# ============================================================
#  FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FAITHFUL XGBOOST - 10-FOLD CV RESULTS")
print("="*60)

if len(micro_f1_scores) > 0:
    print(f"Micro-F1 : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC  : {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")
    
    print("\nPer-fold results:")
    for i, (mf, maf, ra) in enumerate(zip(micro_f1_scores, macro_f1_scores, roc_auc_scores), 1):
        print(f"  Fold {i:2d}: Micro-F1={mf:.4f}, Macro-F1={maf:.4f}, ROC-AUC={ra:.4f}")
else:
    print(" No valid folds completed!")

# ============================================================
#  FEATURE IMPORTANCE ANALYSIS
# ============================================================
print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Train on full dataset for feature importance
X_all_raw, all_valid_idx = extract_features_with_drug_history(
    lab_sequences_by_patient,
    rad_sequences_by_patient,
    drug_desc_sequences_by_patient,
    drug_history_sequences
)

y_all = []
for idx in all_valid_idx:
    seq = labels_by_patient_enc[idx]
    if len(seq) >= 2:
        y_all.append(seq[-1])
y_all = np.array(y_all)

scaler_full = StandardScaler()
X_all = scaler_full.fit_transform(X_all_raw)

dtrain_full = xgb.DMatrix(X_all, label=y_all)
params_full = xgb_params.copy()
params_full['num_class'] = n_classes
params_full['verbosity'] = 0

model_full = xgb.train(params_full, dtrain_full, num_boost_round=100, verbose_eval=False)

# Get feature importance
importance = model_full.get_score(importance_type='gain')
feature_names = [f'f{i}' for i in range(X_all.shape[1])]
importance_dict = dict(zip(feature_names, [0]*len(feature_names)))
importance_dict.update(importance)

# Calculate importance by modality
lab_dim = lab_sequences_by_patient[0].shape[-1]
rad_dim = rad_sequences_by_patient[0].shape[-1]
drug_desc_dim = drug_desc_sequences_by_patient[0].shape[-1]
drug_history_dim = drug_history_sequences[0].shape[-1]

modality_importance = {
    'LAB': sum(importance_dict.get(f'f{i}', 0) for i in range(lab_dim)),
    'RADIOLOGY': sum(importance_dict.get(f'f{i}', 0) for i in range(lab_dim, lab_dim + rad_dim)),
    'DRUG_DESC': sum(importance_dict.get(f'f{i}', 0) for i in range(lab_dim + rad_dim, lab_dim + rad_dim + drug_desc_dim)),
    'DRUG_HISTORY': sum(importance_dict.get(f'f{i}', 0) for i in range(lab_dim + rad_dim + drug_desc_dim, X_all.shape[1]))
}

# Normalize
total_importance = sum(modality_importance.values())
if total_importance > 0:
    modality_importance = {k: v/total_importance for k, v in modality_importance.items()}

print("\nModality contributions to cancer prediction:")
for modality, importance in sorted(modality_importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {modality:15s}: {importance:.2%}")



CREATING DRUG LABELS ENCODING
Number of drugs: 2374
Created drug history for 7011 patients
Example shape: (1, 2374)

Number of cancer classes: 7
Classes: ['BLADDER_CANCER' 'BREAST_CANCER' 'COLORECTAL_CANCER' 'HEAD_NECK_CANCER'
 'LUNG_CANCER' 'OTHER_CANCER' 'PROSTATE_CANCER']

FAITHFUL XGBOOST - MATCHING GRU EXACTLY
✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)
✓ Same patient splitting strategy
✓ Same prediction task (next admission cancer type)
✓ XGBoost parameters:
    n_estimators: 300
    max_depth: 6
    learning_rate: 0.05
    subsample: 0.8
    colsample_bytree: 0.8
    min_child_weight: 3
    gamma: 0.1
    reg_alpha: 0.1
    reg_lambda: 1.0
    objective: multi:softprob
    eval_metric: mlogloss
    random_state: 42
    n_jobs: -1
    verbosity: 0

Drug history sequences prepared: 7011 patients

XGBoost Fold 1/10
Train samples: 1867
Val samples: 456
Test samples: 242
Feature dimension: 10054
Classes in train: [0 1 2 3 4 5 6]


KeyboardInterrupt: 

Logistic Regration Classifier

In [20]:
# ============================================================
#  CREATE DRUG LABELS ENCODING (Matching GRU)
# ============================================================
print("\n" + "="*60)
print("CREATING DRUG LABELS ENCODING")
print("="*60)

# Get number of drugs from DDI matrix
n_drugs = ddi_severity_matrix.shape[0]
print(f"Number of drugs: {n_drugs}")

# Convert drug sequences to multi-hot encoded vectors
drug_labels_by_patient_enc = []

for patient_drug_seq in drug_labels_by_patient:
    # Initialize multi-hot matrix for this patient
    seq_enc = np.zeros((len(patient_drug_seq), n_drugs), dtype=np.float32)
    
    for t, drugs in enumerate(patient_drug_seq):
        if len(drugs) > 0:
            # Ensure drug indices are valid
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    
    drug_labels_by_patient_enc.append(seq_enc)

print(f"Created drug history for {len(drug_labels_by_patient_enc)} patients")
print(f"Example shape: {drug_labels_by_patient_enc[0].shape}")

# ============================================================
#  ENCODE LABELS FOR LOGISTIC REGRESSION
# ============================================================
from sklearn.preprocessing import LabelEncoder

# Encode cancer labels
label_encoder = LabelEncoder()
all_labels = []

# Flatten all patient label sequences to fit encoder
for lbl_seq in labels_by_patient:
    all_labels.extend(lbl_seq)

# Fit encoder on all unique labels
label_encoder.fit(all_labels)

# Transform each patient's labels
labels_by_patient_enc = []
for lbl_seq in labels_by_patient:
    labels_by_patient_enc.append(label_encoder.transform(lbl_seq))

# Get number of classes
n_classes = len(label_encoder.classes_)
print(f"\nNumber of cancer classes: {n_classes}")
print(f"Classes: {label_encoder.classes_}")

# ============================================================
#  FAITHFUL LOGISTIC REGRESSION - MATCHES GRU EXACTLY
# ============================================================

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier

# -----------------------------
# Settings (optimized for Logistic Regression)
# -----------------------------
logreg_params = {
    'C': 1.0,                 # Inverse regularization strength (smaller = stronger regularization)
    'penalty': 'l2',          # L2 regularization
    'solver': 'lbfgs',        # Solver for multiclass
    'max_iter': 1000,         # Maximum iterations for convergence
    'multi_class': 'ovr',     # One-vs-Rest for multiclass
    'class_weight': 'balanced', # Handle class imbalance (matches GRU's loss weights)
    'random_state': 42,
    'n_jobs': -1,
    'tol': 1e-4               # Tolerance for stopping criteria
}

print("\n" + "="*60)
print("FAITHFUL LOGISTIC REGRESSION - MATCHING GRU EXACTLY")
print("="*60)
print("✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)")
print("✓ Same patient splitting strategy")
print("✓ Same prediction task (next admission cancer type)")
print("✓ Logistic Regression parameters:")
for key, value in logreg_params.items():
    print(f"    {key}: {value}")
print("="*60)

# -----------------------------
# Use ALL 4 modalities (match GRU)
# -----------------------------
def extract_features_with_drug_history(lab_seqs, rad_seqs, drug_desc_seqs, drug_history_seqs):
    """
    Extract features including drug history.
    Matches GRU's input exactly.
    """
    features = []
    valid_indices = []
    
    for i, (lab, rad, drug_desc, drug_history) in enumerate(zip(
        lab_seqs, rad_seqs, drug_desc_seqs, drug_history_seqs
    )):
        # Need at least 2 admissions to predict next (same as GRU)
        if len(lab) < 2:
            continue
            
        # Use features from admission t to predict t+1
        x = np.concatenate([
            lab[-2],           # Lab at t
            rad[-2],           # Radiology at t
            drug_desc[-2],     # Drug description at t
            drug_history[-2]   # Drug history at t
        ])
        
        features.append(x)
        valid_indices.append(i)
    
    if len(features) == 0:
        return np.array([]), np.array([])
    
    return np.array(features), np.array(valid_indices)

# -----------------------------
# Prepare drug history as features (multi-hot encoded)
# -----------------------------
drug_history_sequences = []
for i in range(len(drug_labels_by_patient_enc)):
    drug_history_sequences.append(drug_labels_by_patient_enc[i])

print(f"\nDrug history sequences prepared: {len(drug_history_sequences)} patients")

# -----------------------------
# CV Setup (match GRU exactly)
# -----------------------------
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
micro_f1_scores = []
macro_f1_scores = []
roc_auc_scores = []

# ============================================================
#  CROSS-VALIDATION LOOP
# ============================================================
for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    
    print(f"\n{'='*60}")
    print(f"Logistic Regression Fold {fold}/10")
    print(f"{'='*60}")
    
    # Train/val split (match GRU)
    train_idx, val_idx = train_test_split(
        train_idx, 
        test_size=0.2, 
        random_state=42, 
        stratify=[strat_labels[i] for i in train_idx]
    )
    
    # Build TRAIN set with ALL modalities
    train_lab = [lab_sequences_by_patient[i] for i in train_idx]
    train_rad = [rad_sequences_by_patient[i] for i in train_idx]
    train_drug_desc = [drug_desc_sequences_by_patient[i] for i in train_idx]
    train_drug_history = [drug_history_sequences[i] for i in train_idx]
    train_labels = [labels_by_patient_enc[i] for i in train_idx]
    
    X_train_raw, train_valid_idx = extract_features_with_drug_history(
        train_lab, train_rad, train_drug_desc, train_drug_history
    )
    y_train = []
    for idx in train_valid_idx:
        seq = train_labels[idx]
        if len(seq) >= 2:
            y_train.append(seq[-1])
    y_train = np.array(y_train)
    
    if len(X_train_raw) == 0:
        print(f" Warning: No valid training samples in fold {fold}")
        continue
    
    # Build VALIDATION set (for monitoring, no early stopping for LR)
    val_lab = [lab_sequences_by_patient[i] for i in val_idx]
    val_rad = [rad_sequences_by_patient[i] for i in val_idx]
    val_drug_desc = [drug_desc_sequences_by_patient[i] for i in val_idx]
    val_drug_history = [drug_history_sequences[i] for i in val_idx]
    val_labels = [labels_by_patient_enc[i] for i in val_idx]
    
    X_val_raw, val_valid_idx = extract_features_with_drug_history(
        val_lab, val_rad, val_drug_desc, val_drug_history
    )
    y_val = []
    for idx in val_valid_idx:
        seq = val_labels[idx]
        if len(seq) >= 2:
            y_val.append(seq[-1])
    y_val = np.array(y_val)
    
    # Build TEST set with ALL modalities
    test_lab = [lab_sequences_by_patient[i] for i in test_idx]
    test_rad = [rad_sequences_by_patient[i] for i in test_idx]
    test_drug_desc = [drug_desc_sequences_by_patient[i] for i in test_idx]
    test_drug_history = [drug_history_sequences[i] for i in test_idx]
    test_labels = [labels_by_patient_enc[i] for i in test_idx]
    
    X_test_raw, test_valid_idx = extract_features_with_drug_history(
        test_lab, test_rad, test_drug_desc, test_drug_history
    )
    y_test = []
    for idx in test_valid_idx:
        seq = test_labels[idx]
        if len(seq) >= 2:
            y_test.append(seq[-1])
    y_test = np.array(y_test)
    
    if len(X_test_raw) == 0:
        print(f" Warning: No valid test samples in fold {fold}")
        continue
    
    print(f"Train samples: {X_train_raw.shape[0]}")
    print(f"Val samples: {X_val_raw.shape[0]}")
    print(f"Test samples: {X_test_raw.shape[0]}")
    print(f"Feature dimension: {X_train_raw.shape[1]}")
    print(f"Classes in train: {np.unique(y_train)}")
    
    # Scale features (CRITICAL for Logistic Regression!)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)
    
    # Train Logistic Regression
    logreg = LogisticRegression(**logreg_params)
    
    # For multiclass, use OneVsRestClassifier
    if n_classes > 2:
        model = OneVsRestClassifier(logreg, n_jobs=-1)
    else:
        model = logreg
    
    model.fit(X_train, y_train)
    
    # Validation performance (monitoring)
    y_val_pred = model.predict(X_val)
    val_macro_f1 = f1_score(y_val, y_val_pred, average='macro')
    print(f"Validation Macro-F1: {val_macro_f1:.4f}")
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Get probabilities for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
    else:
        # For OneVsRestClassifier
        y_proba = model.predict_proba(X_test)
    
    # Metrics
    micro_f1 = f1_score(y_test, y_pred, average="micro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    
    try:
        if n_classes == 2:
            roc_auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
    except Exception as e:
        print(f"ROC-AUC computation failed: {e}")
        roc_auc = 0.0
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    roc_auc_scores.append(roc_auc)
    
    print(f"\nFold {fold} Results:")
    print(f"  Micro-F1 : {micro_f1:.4f}")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")

# ============================================================
#  FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FAITHFUL LOGISTIC REGRESSION - 10-FOLD CV RESULTS")
print("="*60)

if len(micro_f1_scores) > 0:
    print(f"Micro-F1 : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC  : {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")
    
    print("\nPer-fold results:")
    for i, (mf, maf, ra) in enumerate(zip(micro_f1_scores, macro_f1_scores, roc_auc_scores), 1):
        print(f"  Fold {i:2d}: Micro-F1={mf:.4f}, Macro-F1={maf:.4f}, ROC-AUC={ra:.4f}")
else:
    print(" No valid folds completed!")

# ============================================================
#  COEFFICIENT ANALYSIS (Feature Importance)
# ============================================================
print("\n" + "="*60)
print("COEFFICIENT ANALYSIS (Feature Importance)")
print("="*60)

# Train on full dataset for coefficient analysis
X_all_raw, all_valid_idx = extract_features_with_drug_history(
    lab_sequences_by_patient,
    rad_sequences_by_patient,
    drug_desc_sequences_by_patient,
    drug_history_sequences
)

y_all = []
for idx in all_valid_idx:
    seq = labels_by_patient_enc[idx]
    if len(seq) >= 2:
        y_all.append(seq[-1])
y_all = np.array(y_all)

scaler_full = StandardScaler()
X_all = scaler_full.fit_transform(X_all_raw)

# Train final model
logreg_full = LogisticRegression(**logreg_params)
if n_classes > 2:
    model_full = OneVsRestClassifier(logreg_full, n_jobs=-1)
else:
    model_full = logreg_full

model_full.fit(X_all, y_all)

# Get coefficient magnitudes
if n_classes > 2:
    # For multiclass, get average absolute coefficient across classes
    coef_abs = np.abs(model_full.coef_).mean(axis=0) if hasattr(model_full, 'coef_') else np.zeros(X_all.shape[1])
else:
    coef_abs = np.abs(model_full.coef_[0])

# Calculate importance by modality
lab_dim = lab_sequences_by_patient[0].shape[-1]
rad_dim = rad_sequences_by_patient[0].shape[-1]
drug_desc_dim = drug_desc_sequences_by_patient[0].shape[-1]
drug_history_dim = drug_history_sequences[0].shape[-1]

modality_importance = {
    'LAB': np.sum(coef_abs[:lab_dim]),
    'RADIOLOGY': np.sum(coef_abs[lab_dim:lab_dim + rad_dim]),
    'DRUG_DESC': np.sum(coef_abs[lab_dim + rad_dim:lab_dim + rad_dim + drug_desc_dim]),
    'DRUG_HISTORY': np.sum(coef_abs[lab_dim + rad_dim + drug_desc_dim:])
}

# Normalize
total_importance = sum(modality_importance.values())
if total_importance > 0:
    modality_importance = {k: v/total_importance for k, v in modality_importance.items()}

print("\nModality contributions to cancer prediction (based on coefficient magnitude):")
for modality, importance in sorted(modality_importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {modality:15s}: {importance:.2%}")


CREATING DRUG LABELS ENCODING
Number of drugs: 2374
Created drug history for 7011 patients
Example shape: (1, 2374)

Number of cancer classes: 7
Classes: ['BLADDER_CANCER' 'BREAST_CANCER' 'COLORECTAL_CANCER' 'HEAD_NECK_CANCER'
 'LUNG_CANCER' 'OTHER_CANCER' 'PROSTATE_CANCER']

FAITHFUL LOGISTIC REGRESSION - MATCHING GRU EXACTLY
✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)
✓ Same patient splitting strategy
✓ Same prediction task (next admission cancer type)
✓ Logistic Regression parameters:
    C: 1.0
    penalty: l2
    solver: lbfgs
    max_iter: 1000
    multi_class: ovr
    class_weight: balanced
    random_state: 42
    n_jobs: -1
    tol: 0.0001

Drug history sequences prepared: 7011 patients

Logistic Regression Fold 1/10
Train samples: 1867
Val samples: 456
Test samples: 242
Feature dimension: 10054
Classes in train: [0 1 2 3 4 5 6]
Validation Macro-F1: 0.4673

Fold 1 Results:
  Micro-F1 : 0.5744
  Macro-F1 : 0.4682
  ROC-AUC  : 0.8029

Logistic Regression Fol

GRU Classifier

In [5]:
# ============================================================
#  FAITHFUL GRU CLASSIFIER - CLASSIFICATION ONLY
# Matches Logistic Regression and XGBoost for fair comparison
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score

# -----------------------------
# Device & settings
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32
hidden_dim = 256
num_layers = 2
dropout = 0.1
epochs = 50
learning_rate = 1e-3  # Higher learning rate for GRU convergence
weight_decay = 1e-4

# -----------------------------
# Encode labels (same as before)
# -----------------------------
le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(lbl_seq) for lbl_seq in labels_by_patient]
n_classes = len(le_hf.classes_)
print(f"Number of cancer classes: {n_classes}")
print(f"Classes: {le_hf.classes_}")

# -----------------------------
# Prepare drug history as features (matching RF/XGBoost)
# -----------------------------
# Get number of drugs from DDI matrix
n_drugs = ddi_severity_matrix.shape[0]
print(f"Number of drugs: {n_drugs}")

# Convert drug sequences to multi-hot encoded vectors
drug_history_enc = []
for patient_drug_seq in drug_labels_by_patient:
    seq_enc = np.zeros((len(patient_drug_seq), n_drugs), dtype=np.float32)
    for t, drugs in enumerate(patient_drug_seq):
        if len(drugs) > 0:
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    drug_history_enc.append(seq_enc)

print(f"Drug history encoded for {len(drug_history_enc)} patients")

# -----------------------------
# Dataset (Classification Only - Matches RF/XGBoost)
# -----------------------------
class ClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, lab_sequences, rad_sequences, drug_desc_sequences, 
                 drug_history_sequences, class_labels):
        self.lab_sequences = lab_sequences
        self.rad_sequences = rad_sequences
        self.drug_desc_sequences = drug_desc_sequences
        self.drug_history_sequences = drug_history_sequences
        self.class_labels = class_labels
        
    def __len__(self):
        return len(self.lab_sequences)
    
    def __getitem__(self, idx):
        # Get sequences
        lab_seq = self.lab_sequences[idx]
        rad_seq = self.rad_sequences[idx]
        drug_desc_seq = self.drug_desc_sequences[idx]
        drug_history_seq = self.drug_history_sequences[idx]
        
        # Make sure all sequences have same length
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq))
        
        lab_seq = lab_seq[:min_len]
        rad_seq = rad_seq[:min_len]
        drug_desc_seq = drug_desc_seq[:min_len]
        drug_history_seq = drug_history_seq[:min_len]
        
        return (torch.tensor(lab_seq, dtype=torch.float32),
                torch.tensor(rad_seq, dtype=torch.float32),
                torch.tensor(drug_desc_seq, dtype=torch.float32),
                torch.tensor(drug_history_seq, dtype=torch.float32),  # Multi-hot for GRU
                torch.tensor(self.class_labels[idx][:min_len], dtype=torch.long))

def collate_fn(batch):
    lab_seq, rad_seq, drug_desc, drug_history, ys_class = zip(*batch)
    
    # Get lengths
    lengths = torch.tensor([len(x) for x in lab_seq], dtype=torch.long)
    
    # Pad sequences
    lab_padded = pad_sequence(lab_seq, batch_first=True, padding_value=0.0)
    rad_padded = pad_sequence(rad_seq, batch_first=True, padding_value=0.0)
    drug_desc_padded = pad_sequence(drug_desc, batch_first=True, padding_value=0.0)
    drug_history_padded = pad_sequence(drug_history, batch_first=True, padding_value=0.0)
    cls_padded = pad_sequence(ys_class, batch_first=True, padding_value=-100)
    
    return (lab_padded, rad_padded, drug_desc_padded, drug_history_padded,
            cls_padded, lengths)

# -----------------------------
# FAITHFUL GRU CLASSIFIER (Replaces Transformer)
# -----------------------------
class FaithfulGRUClassifier(nn.Module):
    def __init__(self, lab_dim, rad_dim, drug_desc_dim, drug_history_dim, n_classes,
                 hidden_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        
        # Input projections for each modality
        self.lab_proj = nn.Sequential(
            nn.Linear(lab_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.rad_proj = nn.Sequential(
            nn.Linear(rad_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.drug_desc_proj = nn.Sequential(
            nn.Linear(drug_desc_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.drug_history_proj = nn.Sequential(
            nn.Linear(drug_history_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Fusion layer (simple concatenation + projection)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # GRU for sequence modeling (autoregressive by nature)
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False  # Causal/autoregressive
        )
        
        # Layer norm after GRU
        self.gru_norm = nn.LayerNorm(hidden_dim)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes)
        )
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight, gain=0.5)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0.0)
        elif isinstance(module, nn.GRU):
            for name, param in module.named_parameters():
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(param)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(param)
                elif 'bias' in name:
                    nn.init.constant_(param, 0.0)
                    
    def forward(self, lab_x, rad_x, drug_desc_x, drug_history_x, lengths):
        # Project each modality
        lab_h = self.lab_proj(lab_x)
        rad_h = self.rad_proj(rad_x)
        drug_desc_h = self.drug_desc_proj(drug_desc_x)
        drug_history_h = self.drug_history_proj(drug_history_x)
        
        # Concatenate all modalities
        combined = torch.cat([lab_h, rad_h, drug_desc_h, drug_history_h], dim=-1)
        
        # Fuse modalities
        fused = self.fusion(combined)
        
        # Pack padded sequence for GRU
        packed_input = pack_padded_sequence(fused, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # GRU forward pass (naturally causal/autoregressive)
        packed_output, hidden = self.gru(packed_input)
        
        # Unpack
        gru_out, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        # Apply layer norm and dropout
        out = self.gru_norm(gru_out)
        out = self.dropout(out)
        
        # Classification logits
        logits = self.classifier(out)
        
        return logits

# -----------------------------
# Training function
# -----------------------------
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n_batches = 0
    
    for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in loader:
        lab_x = lab_x.to(device)
        rad_x = rad_x.to(device)
        drug_desc_x = drug_desc_x.to(device)
        drug_history_x = drug_history_x.to(device)
        y_cls = y_cls.to(device)
        lengths = lengths.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
        
        # Compute loss (predict next admission)
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, n_classes),
            y_cls[:, 1:].reshape(-1),
            ignore_index=-100
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches

# -----------------------------
# Evaluation function
# -----------------------------
def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_proba = []
    
    with torch.no_grad():
        for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in loader:
            lab_x = lab_x.to(device)
            rad_x = rad_x.to(device)
            drug_desc_x = drug_desc_x.to(device)
            drug_history_x = drug_history_x.to(device)
            y_cls = y_cls.to(device)
            lengths = lengths.to(device)
            
            logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
            probs = F.softmax(logits, dim=-1)
            
            for i in range(len(lengths)):
                L = lengths[i].item()
                if L <= 1:
                    continue
                
                # Predict next admission (t+1) from current (t)
                y_true.extend(y_cls[i, 1:L].cpu().numpy())
                y_pred.extend(probs[i, :-1][:L-1].argmax(-1).cpu().numpy())
                y_proba.extend(probs[i, :-1][:L-1].cpu().numpy())
    
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_proba = np.array(y_proba)
    
    micro_f1 = f1_score(y_true, y_pred, average="micro")
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except:
        roc_auc = 0.0
    
    return micro_f1, macro_f1, roc_auc

# ============================================================
#  CROSS-VALIDATION LOOP
# ============================================================

print("\n" + "="*60)
print("FAITHFUL GRU CLASSIFIER - CLASSIFICATION ONLY")
print("="*60)
print("✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)")
print("✓ Same patient splitting strategy as RF/XGBoost")
print("✓ Same prediction task (next admission cancer type)")
print("✓ Classification only (no drug prediction)")
print("="*60)

# CV setup
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
micro_f1_scores = []
macro_f1_scores = []
roc_auc_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    print(f"\n{'='*60}")
    print(f"GRU Fold {fold}/10")
    print(f"{'='*60}")
    
    # Train/val split
    train_idx, val_idx = train_test_split(
        train_idx, 
        test_size=0.2, 
        random_state=42, 
        stratify=[strat_labels[i] for i in train_idx]
    )
    
    # Get dimensions
    lab_dim = lab_sequences_by_patient[0].shape[-1]
    rad_dim = rad_sequences_by_patient[0].shape[-1]
    drug_desc_dim = drug_desc_sequences_by_patient[0].shape[-1]
    drug_history_dim = drug_history_enc[0].shape[-1]
    
    # Prepare training data
    train_lab = [lab_sequences_by_patient[i] for i in train_idx]
    train_rad = [rad_sequences_by_patient[i] for i in train_idx]
    train_drug_desc = [drug_desc_sequences_by_patient[i] for i in train_idx]
    train_drug_history = [drug_history_enc[i] for i in train_idx]
    train_labels = [labels_by_patient_enc[i] for i in train_idx]
    
    # Prepare validation data
    val_lab = [lab_sequences_by_patient[i] for i in val_idx]
    val_rad = [rad_sequences_by_patient[i] for i in val_idx]
    val_drug_desc = [drug_desc_sequences_by_patient[i] for i in val_idx]
    val_drug_history = [drug_history_enc[i] for i in val_idx]
    val_labels = [labels_by_patient_enc[i] for i in val_idx]
    
    # Prepare test data
    test_lab = [lab_sequences_by_patient[i] for i in test_idx]
    test_rad = [rad_sequences_by_patient[i] for i in test_idx]
    test_drug_desc = [drug_desc_sequences_by_patient[i] for i in test_idx]
    test_drug_history = [drug_history_enc[i] for i in test_idx]
    test_labels = [labels_by_patient_enc[i] for i in test_idx]
    
    # Create datasets
    train_dataset = ClassificationDataset(
        train_lab, train_rad, train_drug_desc, train_drug_history, train_labels
    )
    val_dataset = ClassificationDataset(
        val_lab, val_rad, val_drug_desc, val_drug_history, val_labels
    )
    test_dataset = ClassificationDataset(
        test_lab, test_rad, test_drug_desc, test_drug_history, test_labels
    )
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                            collate_fn=collate_fn, num_workers=0)
    
    # Initialize model
    model = FaithfulGRUClassifier(
        lab_dim=lab_dim,
        rad_dim=rad_dim,
        drug_desc_dim=drug_desc_dim,
        drug_history_dim=drug_history_dim,
        n_classes=n_classes,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)
    
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")
    print(f"Input dimensions: Lab={lab_dim}, Rad={rad_dim}, DrugDesc={drug_desc_dim}, DrugHistory={drug_history_dim}")
    
    # Optimizer with GRU-specific settings (higher LR)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Training
    best_val_loss = float('inf')
    patience = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer)
        
        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in val_loader:
                lab_x = lab_x.to(device)
                rad_x = rad_x.to(device)
                drug_desc_x = drug_desc_x.to(device)
                drug_history_x = drug_history_x.to(device)
                y_cls = y_cls.to(device)
                lengths = lengths.to(device)
                
                logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
                loss = F.cross_entropy(
                    logits[:, :-1].reshape(-1, n_classes),
                    y_cls[:, 1:].reshape(-1),
                    ignore_index=-100
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= 7:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(best_model_state)
    model = model.to(device)
    
    # Evaluate
    micro_f1, macro_f1, roc_auc = evaluate(model, test_loader)
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    roc_auc_scores.append(roc_auc)
    
    print(f"\nFold {fold} Results:")
    print(f"  Micro-F1 : {micro_f1:.4f}")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")

# ============================================================
# FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FAITHFUL GRU CLASSIFIER - 10-FOLD CV RESULTS")
print("="*60)

if len(micro_f1_scores) > 0:
    print(f"Micro-F1 : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC  : {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")
    
    print("\nPer-fold results:")
    for i, (mf, maf, ra) in enumerate(zip(micro_f1_scores, macro_f1_scores, roc_auc_scores), 1):
        print(f"  Fold {i:2d}: Micro-F1={mf:.4f}, Macro-F1={maf:.4f}, ROC-AUC={ra:.4f}")
else:
    print(" No valid folds completed!")

Number of cancer classes: 7
Classes: ['BLADDER_CANCER' 'BREAST_CANCER' 'COLORECTAL_CANCER' 'HEAD_NECK_CANCER'
 'LUNG_CANCER' 'OTHER_CANCER' 'PROSTATE_CANCER']
Number of drugs: 2374
Drug history encoded for 7011 patients

FAITHFUL GRU CLASSIFIER - CLASSIFICATION ONLY
✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)
✓ Same patient splitting strategy as RF/XGBoost
✓ Same prediction task (next admission cancer type)
✓ Classification only (no drug prediction)

GRU Fold 1/10
Trainable parameters: 3,663,623
Input dimensions: Lab=2560, Rad=2560, DrugDesc=2560, DrugHistory=2374
Epoch  5/50 | Train Loss: 1.3084 | Val Loss: 1.3548
Epoch 10/50 | Train Loss: 0.9298 | Val Loss: 1.9584
Early stopping at epoch 10

Fold 1 Results:
  Micro-F1 : 0.5360
  Macro-F1 : 0.1644
  ROC-AUC  : 0.7282

GRU Fold 2/10
Trainable parameters: 3,663,623
Input dimensions: Lab=2560, Rad=2560, DrugDesc=2560, DrugHistory=2374
Epoch  5/50 | Train Loss: 1.3051 | Val Loss: 1.2746
Epoch 10/50 | Train Loss: 0.9163

Transformer Classifier

In [25]:
# ============================================================
#  FAITHFUL TRANSFORMER CLASSIFIER - CLASSIFICATION ONLY
# Matches Logistic Regression and XGBoost for fair comparison
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score

# -----------------------------
# Device & settings
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32
hidden_dim = 256
num_layers = 2
num_heads = 4
dropout = 0.1
epochs = 50
learning_rate = 1e-3  # Higher learning rate for transformer convergence
weight_decay = 1e-4

# -----------------------------
# Encode labels (same as before)
# -----------------------------
le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(lbl_seq) for lbl_seq in labels_by_patient]
n_classes = len(le_hf.classes_)
print(f"Number of cancer classes: {n_classes}")
print(f"Classes: {le_hf.classes_}")

# -----------------------------
# Prepare drug history as features (matching RF/XGBoost)
# -----------------------------
# Get number of drugs from DDI matrix
n_drugs = ddi_severity_matrix.shape[0]
print(f"Number of drugs: {n_drugs}")

# Convert drug sequences to multi-hot encoded vectors
drug_history_enc = []
for patient_drug_seq in drug_labels_by_patient:
    seq_enc = np.zeros((len(patient_drug_seq), n_drugs), dtype=np.float32)
    for t, drugs in enumerate(patient_drug_seq):
        if len(drugs) > 0:
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    drug_history_enc.append(seq_enc)

print(f"Drug history encoded for {len(drug_history_enc)} patients")

# -----------------------------
# Dataset (Classification Only - Matches RF/XGBoost)
# -----------------------------
class ClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, lab_sequences, rad_sequences, drug_desc_sequences, 
                 drug_history_sequences, class_labels):
        self.lab_sequences = lab_sequences
        self.rad_sequences = rad_sequences
        self.drug_desc_sequences = drug_desc_sequences
        self.drug_history_sequences = drug_history_sequences
        self.class_labels = class_labels
        
    def __len__(self):
        return len(self.lab_sequences)
    
    def __getitem__(self, idx):
        # Get sequences
        lab_seq = self.lab_sequences[idx]
        rad_seq = self.rad_sequences[idx]
        drug_desc_seq = self.drug_desc_sequences[idx]
        drug_history_seq = self.drug_history_sequences[idx]
        
        # Make sure all sequences have same length
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq))
        
        lab_seq = lab_seq[:min_len]
        rad_seq = rad_seq[:min_len]
        drug_desc_seq = drug_desc_seq[:min_len]
        drug_history_seq = drug_history_seq[:min_len]
        
        return (torch.tensor(lab_seq, dtype=torch.float32),
                torch.tensor(rad_seq, dtype=torch.float32),
                torch.tensor(drug_desc_seq, dtype=torch.float32),
                torch.tensor(drug_history_seq, dtype=torch.float32),  # Multi-hot for transformer
                torch.tensor(self.class_labels[idx][:min_len], dtype=torch.long))

def collate_fn(batch):
    lab_seq, rad_seq, drug_desc, drug_history, ys_class = zip(*batch)
    
    # Get lengths
    lengths = torch.tensor([len(x) for x in lab_seq], dtype=torch.long)
    
    # Pad sequences
    lab_padded = pad_sequence(lab_seq, batch_first=True, padding_value=0.0)
    rad_padded = pad_sequence(rad_seq, batch_first=True, padding_value=0.0)
    drug_desc_padded = pad_sequence(drug_desc, batch_first=True, padding_value=0.0)
    drug_history_padded = pad_sequence(drug_history, batch_first=True, padding_value=0.0)
    cls_padded = pad_sequence(ys_class, batch_first=True, padding_value=-100)
    
    return (lab_padded, rad_padded, drug_desc_padded, drug_history_padded,
            cls_padded, lengths)

# -----------------------------
# Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term[:pe[:, 0::2].shape[1]])
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# -----------------------------
# FAITHFUL TRANSFORMER CLASSIFIER (Replaces GRU)
# -----------------------------
class FaithfulTransformerClassifier(nn.Module):
    def __init__(self, lab_dim, rad_dim, drug_desc_dim, drug_history_dim, n_classes,
                 hidden_dim=256, num_layers=2, num_heads=4, dropout=0.1):
        super().__init__()
        
        # Input projections for each modality
        self.lab_proj = nn.Sequential(
            nn.Linear(lab_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.rad_proj = nn.Sequential(
            nn.Linear(rad_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.drug_desc_proj = nn.Sequential(
            nn.Linear(drug_desc_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.drug_history_proj = nn.Sequential(
            nn.Linear(drug_history_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Fusion layer (simple concatenation + projection)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(hidden_dim)
        
        # Transformer Encoder (replaces GRU)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Layer norm after transformer
        self.transformer_norm = nn.LayerNorm(hidden_dim)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes)
        )
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight, gain=0.5)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0.0)
        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)
                    
    def forward(self, lab_x, rad_x, drug_desc_x, drug_history_x, lengths):
        # Project each modality
        lab_h = self.lab_proj(lab_x)
        rad_h = self.rad_proj(rad_x)
        drug_desc_h = self.drug_desc_proj(drug_desc_x)
        drug_history_h = self.drug_history_proj(drug_history_x)
        
        # Concatenate all modalities
        combined = torch.cat([lab_h, rad_h, drug_desc_h, drug_history_h], dim=-1)
        
        # Fuse modalities
        fused = self.fusion(combined)
        
        # Add positional encoding
        fused = self.pos_encoder(fused)
        
        # Create padding mask for transformer
        padding_mask = torch.arange(fused.shape[1], device=lengths.device).expand(len(lengths), fused.shape[1]) >= lengths.unsqueeze(1)
        
        # Transformer forward pass (replaces GRU)
        transformer_out = self.transformer(fused, src_key_padding_mask=padding_mask)
        
        # Apply layer norm and dropout
        out = self.transformer_norm(transformer_out)
        out = self.dropout(out)
        
        # Classification logits
        logits = self.classifier(out)
        
        return logits

# -----------------------------
# Training function
# -----------------------------
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n_batches = 0
    
    for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in loader:
        lab_x = lab_x.to(device)
        rad_x = rad_x.to(device)
        drug_desc_x = drug_desc_x.to(device)
        drug_history_x = drug_history_x.to(device)
        y_cls = y_cls.to(device)
        lengths = lengths.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
        
        # Compute loss (predict next admission)
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, n_classes),
            y_cls[:, 1:].reshape(-1),
            ignore_index=-100
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches

# -----------------------------
# Evaluation function
# -----------------------------
def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_proba = []
    
    with torch.no_grad():
        for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in loader:
            lab_x = lab_x.to(device)
            rad_x = rad_x.to(device)
            drug_desc_x = drug_desc_x.to(device)
            drug_history_x = drug_history_x.to(device)
            y_cls = y_cls.to(device)
            lengths = lengths.to(device)
            
            logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
            probs = F.softmax(logits, dim=-1)
            
            for i in range(len(lengths)):
                L = lengths[i].item()
                if L <= 1:
                    continue
                
                # Predict next admission (t+1) from current (t)
                y_true.extend(y_cls[i, 1:L].cpu().numpy())
                y_pred.extend(probs[i, :-1][:L-1].argmax(-1).cpu().numpy())
                y_proba.extend(probs[i, :-1][:L-1].cpu().numpy())
    
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_proba = np.array(y_proba)
    
    micro_f1 = f1_score(y_true, y_pred, average="micro")
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except:
        roc_auc = 0.0
    
    return micro_f1, macro_f1, roc_auc

# ============================================================
#  CROSS-VALIDATION LOOP
# ============================================================

print("\n" + "="*60)
print("FAITHFUL TRANSFORMER CLASSIFIER - CLASSIFICATION ONLY")
print("="*60)
print("✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)")
print("✓ Same patient splitting strategy as RF/XGBoost")
print("✓ Same prediction task (next admission cancer type)")
print("✓ Classification only (no drug prediction)")
print("="*60)

# CV setup
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
micro_f1_scores = []
macro_f1_scores = []
roc_auc_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    print(f"\n{'='*60}")
    print(f"Transformer Fold {fold}/10")
    print(f"{'='*60}")
    
    # Train/val split
    train_idx, val_idx = train_test_split(
        train_idx, 
        test_size=0.2, 
        random_state=42, 
        stratify=[strat_labels[i] for i in train_idx]
    )
    
    # Get dimensions
    lab_dim = lab_sequences_by_patient[0].shape[-1]
    rad_dim = rad_sequences_by_patient[0].shape[-1]
    drug_desc_dim = drug_desc_sequences_by_patient[0].shape[-1]
    drug_history_dim = drug_history_enc[0].shape[-1]
    
    # Prepare training data
    train_lab = [lab_sequences_by_patient[i] for i in train_idx]
    train_rad = [rad_sequences_by_patient[i] for i in train_idx]
    train_drug_desc = [drug_desc_sequences_by_patient[i] for i in train_idx]
    train_drug_history = [drug_history_enc[i] for i in train_idx]
    train_labels = [labels_by_patient_enc[i] for i in train_idx]
    
    # Prepare validation data
    val_lab = [lab_sequences_by_patient[i] for i in val_idx]
    val_rad = [rad_sequences_by_patient[i] for i in val_idx]
    val_drug_desc = [drug_desc_sequences_by_patient[i] for i in val_idx]
    val_drug_history = [drug_history_enc[i] for i in val_idx]
    val_labels = [labels_by_patient_enc[i] for i in val_idx]
    
    # Prepare test data
    test_lab = [lab_sequences_by_patient[i] for i in test_idx]
    test_rad = [rad_sequences_by_patient[i] for i in test_idx]
    test_drug_desc = [drug_desc_sequences_by_patient[i] for i in test_idx]
    test_drug_history = [drug_history_enc[i] for i in test_idx]
    test_labels = [labels_by_patient_enc[i] for i in test_idx]
    
    # Create datasets
    train_dataset = ClassificationDataset(
        train_lab, train_rad, train_drug_desc, train_drug_history, train_labels
    )
    val_dataset = ClassificationDataset(
        val_lab, val_rad, val_drug_desc, val_drug_history, val_labels
    )
    test_dataset = ClassificationDataset(
        test_lab, test_rad, test_drug_desc, test_drug_history, test_labels
    )
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                            collate_fn=collate_fn, num_workers=0)
    
    # Initialize model
    model = FaithfulTransformerClassifier(
        lab_dim=lab_dim,
        rad_dim=rad_dim,
        drug_desc_dim=drug_desc_dim,
        drug_history_dim=drug_history_dim,
        n_classes=n_classes,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_heads=num_heads,
        dropout=dropout
    ).to(device)
    
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")
    print(f"Input dimensions: Lab={lab_dim}, Rad={rad_dim}, DrugDesc={drug_desc_dim}, DrugHistory={drug_history_dim}")
    
    # Optimizer with transformer-specific settings
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Training
    best_val_loss = float('inf')
    patience = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer)
        
        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in val_loader:
                lab_x = lab_x.to(device)
                rad_x = rad_x.to(device)
                drug_desc_x = drug_desc_x.to(device)
                drug_history_x = drug_history_x.to(device)
                y_cls = y_cls.to(device)
                lengths = lengths.to(device)
                
                logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
                loss = F.cross_entropy(
                    logits[:, :-1].reshape(-1, n_classes),
                    y_cls[:, 1:].reshape(-1),
                    ignore_index=-100
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= 7:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(best_model_state)
    model = model.to(device)
    
    # Evaluate
    micro_f1, macro_f1, roc_auc = evaluate(model, test_loader)
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    roc_auc_scores.append(roc_auc)
    
    print(f"\nFold {fold} Results:")
    print(f"  Micro-F1 : {micro_f1:.4f}")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")

# ============================================================
#  FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FAITHFUL TRANSFORMER CLASSIFIER - 10-FOLD CV RESULTS")
print("="*60)

if len(micro_f1_scores) > 0:
    print(f"Micro-F1 : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC  : {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")
    
    print("\nPer-fold results:")
    for i, (mf, maf, ra) in enumerate(zip(micro_f1_scores, macro_f1_scores, roc_auc_scores), 1):
        print(f"  Fold {i:2d}: Micro-F1={mf:.4f}, Macro-F1={maf:.4f}, ROC-AUC={ra:.4f}")
else:
    print(" No valid folds completed!")



Number of cancer classes: 7
Classes: ['BLADDER_CANCER' 'BREAST_CANCER' 'COLORECTAL_CANCER' 'HEAD_NECK_CANCER'
 'LUNG_CANCER' 'OTHER_CANCER' 'PROSTATE_CANCER']
Number of drugs: 2374
Drug history encoded for 7011 patients

FAITHFUL TRANSFORMER CLASSIFIER - CLASSIFICATION ONLY
✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)
✓ Same patient splitting strategy as RF/XGBoost
✓ Same prediction task (next admission cancer type)
✓ Classification only (no drug prediction)

Transformer Fold 1/10
Trainable parameters: 4,453,639
Input dimensions: Lab=2560, Rad=2560, DrugDesc=2560, DrugHistory=2374
Epoch  5/50 | Train Loss: 1.3222 | Val Loss: 1.2952
Epoch 10/50 | Train Loss: 1.1056 | Val Loss: 1.9780
Early stopping at epoch 12

Fold 1 Results:
  Micro-F1 : 0.5012
  Macro-F1 : 0.1905
  ROC-AUC  : 0.7075

Transformer Fold 2/10
Trainable parameters: 4,453,639
Input dimensions: Lab=2560, Rad=2560, DrugDesc=2560, DrugHistory=2374
Epoch  5/50 | Train Loss: 1.2603 | Val Loss: 1.2686
Epoch 10

LSTM Classifier

In [4]:
# ============================================================
# FAITHFUL LSTM CLASSIFIER - CLASSIFICATION ONLY
# Matches Logistic Regression and XGBoost for fair comparison
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score

# -----------------------------
# Device & settings
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32
hidden_dim = 256
num_layers = 2
dropout = 0.1
epochs = 50
learning_rate = 1e-3
weight_decay = 1e-4

# -----------------------------
# Encode labels
# -----------------------------
le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(lbl_seq) for lbl_seq in labels_by_patient]
n_classes = len(le_hf.classes_)
print(f"Number of cancer classes: {n_classes}")
print(f"Classes: {le_hf.classes_}")

# -----------------------------
# Prepare drug history as features (matching RF/XGBoost)
# -----------------------------
n_drugs = ddi_severity_matrix.shape[0]
print(f"Number of drugs: {n_drugs}")

# Convert drug sequences to multi-hot encoded vectors
drug_history_enc = []
for patient_drug_seq in drug_labels_by_patient:
    seq_enc = np.zeros((len(patient_drug_seq), n_drugs), dtype=np.float32)
    for t, drugs in enumerate(patient_drug_seq):
        if len(drugs) > 0:
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    drug_history_enc.append(seq_enc)

print(f"Drug history encoded for {len(drug_history_enc)} patients")

# -----------------------------
# Dataset (Classification Only)
# -----------------------------
class ClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, lab_sequences, rad_sequences, drug_desc_sequences, 
                 drug_history_sequences, class_labels):
        self.lab_sequences = lab_sequences
        self.rad_sequences = rad_sequences
        self.drug_desc_sequences = drug_desc_sequences
        self.drug_history_sequences = drug_history_sequences
        self.class_labels = class_labels
        
    def __len__(self):
        return len(self.lab_sequences)
    
    def __getitem__(self, idx):
        lab_seq = self.lab_sequences[idx]
        rad_seq = self.rad_sequences[idx]
        drug_desc_seq = self.drug_desc_sequences[idx]
        drug_history_seq = self.drug_history_sequences[idx]
        
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq))
        
        lab_seq = lab_seq[:min_len]
        rad_seq = rad_seq[:min_len]
        drug_desc_seq = drug_desc_seq[:min_len]
        drug_history_seq = drug_history_seq[:min_len]
        
        return (torch.tensor(lab_seq, dtype=torch.float32),
                torch.tensor(rad_seq, dtype=torch.float32),
                torch.tensor(drug_desc_seq, dtype=torch.float32),
                torch.tensor(drug_history_seq, dtype=torch.float32),
                torch.tensor(self.class_labels[idx][:min_len], dtype=torch.long))

def collate_fn(batch):
    lab_seq, rad_seq, drug_desc, drug_history, ys_class = zip(*batch)
    
    lengths = torch.tensor([len(x) for x in lab_seq], dtype=torch.long)
    
    lab_padded = pad_sequence(lab_seq, batch_first=True, padding_value=0.0)
    rad_padded = pad_sequence(rad_seq, batch_first=True, padding_value=0.0)
    drug_desc_padded = pad_sequence(drug_desc, batch_first=True, padding_value=0.0)
    drug_history_padded = pad_sequence(drug_history, batch_first=True, padding_value=0.0)
    cls_padded = pad_sequence(ys_class, batch_first=True, padding_value=-100)
    
    return (lab_padded, rad_padded, drug_desc_padded, drug_history_padded,
            cls_padded, lengths)

# -----------------------------
# FAITHFUL LSTM CLASSIFIER (Replaces Transformer)
# -----------------------------
class FaithfulLSTMClassifier(nn.Module):
    def __init__(self, lab_dim, rad_dim, drug_desc_dim, drug_history_dim, n_classes,
                 hidden_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        
        # Input projections for each modality
        self.lab_proj = nn.Sequential(
            nn.Linear(lab_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.rad_proj = nn.Sequential(
            nn.Linear(rad_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.drug_desc_proj = nn.Sequential(
            nn.Linear(drug_desc_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.drug_history_proj = nn.Sequential(
            nn.Linear(drug_history_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Fusion layer (concatenation + projection)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # LSTM layers (replaces transformer)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0, bidirectional=False)
        self.lstm_norm = nn.LayerNorm(hidden_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes)
        )
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight, gain=0.5)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0.0)
        elif isinstance(module, nn.LSTM):
            for name, param in module.named_parameters():
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(param)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(param)
                elif 'bias' in name:
                    n = param.size(0)
                    start, end = n // 4, n // 2
                    param.data[start:end].fill_(1.0)
                    param.data[:start].fill_(0.0)
                    param.data[end:].fill_(0.0)
                    
    def forward(self, lab_x, rad_x, drug_desc_x, drug_history_x, lengths):
        # Project each modality
        lab_h = self.lab_proj(lab_x)
        rad_h = self.rad_proj(rad_x)
        drug_desc_h = self.drug_desc_proj(drug_desc_x)
        drug_history_h = self.drug_history_proj(drug_history_x)
        
        # Concatenate all modalities
        combined = torch.cat([lab_h, rad_h, drug_desc_h, drug_history_h], dim=-1)
        
        # Fuse modalities
        fused = self.fusion(combined)
        
        # Pack sequences for LSTM
        packed_input = pack_padded_sequence(fused, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # Initialize LSTM hidden states
        h0 = torch.zeros(self.lstm.num_layers, fused.size(0), self.lstm.hidden_size, device=fused.device)
        c0 = torch.zeros(self.lstm.num_layers, fused.size(0), self.lstm.hidden_size, device=fused.device)
        
        # LSTM forward pass
        packed_output, _ = self.lstm(packed_input, (h0, c0))
        lstm_out, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        # Layer norm and dropout
        out = self.lstm_norm(lstm_out)
        out = self.dropout(out)
        
        # Classification logits
        logits = self.classifier(out)
        
        return logits

# -----------------------------
# Training function
# -----------------------------
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n_batches = 0
    
    for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in loader:
        lab_x = lab_x.to(device)
        rad_x = rad_x.to(device)
        drug_desc_x = drug_desc_x.to(device)
        drug_history_x = drug_history_x.to(device)
        y_cls = y_cls.to(device)
        lengths = lengths.to(device)
        
        optimizer.zero_grad()
        
        logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
        
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, n_classes),
            y_cls[:, 1:].reshape(-1),
            ignore_index=-100
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches

# -----------------------------
# Evaluation function
# -----------------------------
def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_proba = []
    
    with torch.no_grad():
        for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in loader:
            lab_x = lab_x.to(device)
            rad_x = rad_x.to(device)
            drug_desc_x = drug_desc_x.to(device)
            drug_history_x = drug_history_x.to(device)
            y_cls = y_cls.to(device)
            lengths = lengths.to(device)
            
            logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
            probs = F.softmax(logits, dim=-1)
            
            for i in range(len(lengths)):
                L = lengths[i].item()
                if L <= 1:
                    continue
                
                y_true.extend(y_cls[i, 1:L].cpu().numpy())
                y_pred.extend(probs[i, :-1][:L-1].argmax(-1).cpu().numpy())
                y_proba.extend(probs[i, :-1][:L-1].cpu().numpy())
    
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_proba = np.array(y_proba)
    
    micro_f1 = f1_score(y_true, y_pred, average="micro")
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except:
        roc_auc = 0.0
    
    return micro_f1, macro_f1, roc_auc

# ============================================================
#  CROSS-VALIDATION LOOP
# ============================================================

print("\n" + "="*60)
print("FAITHFUL LSTM CLASSIFIER - CLASSIFICATION ONLY")
print("="*60)
print("✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)")
print("✓ Same patient splitting strategy as RF/XGBoost")
print("✓ Same prediction task (next admission cancer type)")
print("✓ Classification only (no drug prediction)")
print("="*60)

# CV setup
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
micro_f1_scores = []
macro_f1_scores = []
roc_auc_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    print(f"\n{'='*60}")
    print(f"LSTM Fold {fold}/10")
    print(f"{'='*60}")
    
    train_idx, val_idx = train_test_split(
        train_idx, 
        test_size=0.2, 
        random_state=42, 
        stratify=[strat_labels[i] for i in train_idx]
    )
    
    lab_dim = lab_sequences_by_patient[0].shape[-1]
    rad_dim = rad_sequences_by_patient[0].shape[-1]
    drug_desc_dim = drug_desc_sequences_by_patient[0].shape[-1]
    drug_history_dim = drug_history_enc[0].shape[-1]
    
    # Prepare data
    train_lab = [lab_sequences_by_patient[i] for i in train_idx]
    train_rad = [rad_sequences_by_patient[i] for i in train_idx]
    train_drug_desc = [drug_desc_sequences_by_patient[i] for i in train_idx]
    train_drug_history = [drug_history_enc[i] for i in train_idx]
    train_labels = [labels_by_patient_enc[i] for i in train_idx]
    
    val_lab = [lab_sequences_by_patient[i] for i in val_idx]
    val_rad = [rad_sequences_by_patient[i] for i in val_idx]
    val_drug_desc = [drug_desc_sequences_by_patient[i] for i in val_idx]
    val_drug_history = [drug_history_enc[i] for i in val_idx]
    val_labels = [labels_by_patient_enc[i] for i in val_idx]
    
    test_lab = [lab_sequences_by_patient[i] for i in test_idx]
    test_rad = [rad_sequences_by_patient[i] for i in test_idx]
    test_drug_desc = [drug_desc_sequences_by_patient[i] for i in test_idx]
    test_drug_history = [drug_history_enc[i] for i in test_idx]
    test_labels = [labels_by_patient_enc[i] for i in test_idx]
    
    # Create datasets
    train_dataset = ClassificationDataset(
        train_lab, train_rad, train_drug_desc, train_drug_history, train_labels
    )
    val_dataset = ClassificationDataset(
        val_lab, val_rad, val_drug_desc, val_drug_history, val_labels
    )
    test_dataset = ClassificationDataset(
        test_lab, test_rad, test_drug_desc, test_drug_history, test_labels
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                            collate_fn=collate_fn, num_workers=0)
    
    # Initialize LSTM model
    model = FaithfulLSTMClassifier(
        lab_dim=lab_dim,
        rad_dim=rad_dim,
        drug_desc_dim=drug_desc_dim,
        drug_history_dim=drug_history_dim,
        n_classes=n_classes,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)
    
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")
    print(f"Input dimensions: Lab={lab_dim}, Rad={rad_dim}, DrugDesc={drug_desc_dim}, DrugHistory={drug_history_dim}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Training
    best_val_loss = float('inf')
    patience = 0
    best_model_state = None
    
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for lab_x, rad_x, drug_desc_x, drug_history_x, y_cls, lengths in val_loader:
                lab_x = lab_x.to(device)
                rad_x = rad_x.to(device)
                drug_desc_x = drug_desc_x.to(device)
                drug_history_x = drug_history_x.to(device)
                y_cls = y_cls.to(device)
                lengths = lengths.to(device)
                
                logits = model(lab_x, rad_x, drug_desc_x, drug_history_x, lengths)
                loss = F.cross_entropy(
                    logits[:, :-1].reshape(-1, n_classes),
                    y_cls[:, 1:].reshape(-1),
                    ignore_index=-100
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= 7:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(best_model_state)
    model = model.to(device)
    
    # Evaluate
    micro_f1, macro_f1, roc_auc = evaluate(model, test_loader)
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    roc_auc_scores.append(roc_auc)
    
    print(f"\nFold {fold} Results:")
    print(f"  Micro-F1 : {micro_f1:.4f}")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")

# ============================================================
# FINAL RESULTS
# ============================================================
print("\n" + "="*60)
print("FAITHFUL LSTM CLASSIFIER - 10-FOLD CV RESULTS")
print("="*60)

if len(micro_f1_scores) > 0:
    print(f"Micro-F1 : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC  : {np.mean(roc_auc_scores):.4f} ± {np.std(roc_auc_scores):.4f}")
    
    print("\nPer-fold results:")
    for i, (mf, maf, ra) in enumerate(zip(micro_f1_scores, macro_f1_scores, roc_auc_scores), 1):
        print(f"  Fold {i:2d}: Micro-F1={mf:.4f}, Macro-F1={maf:.4f}, ROC-AUC={ra:.4f}")
else:
    print(" No valid folds completed!")

Number of cancer classes: 7
Classes: ['BLADDER_CANCER' 'BREAST_CANCER' 'COLORECTAL_CANCER' 'HEAD_NECK_CANCER'
 'LUNG_CANCER' 'OTHER_CANCER' 'PROSTATE_CANCER']
Number of drugs: 2374
Drug history encoded for 7011 patients

FAITHFUL LSTM CLASSIFIER - CLASSIFICATION ONLY
✓ Using ALL 4 modalities (Lab + Rad + Drug Desc + Drug History)
✓ Same patient splitting strategy as RF/XGBoost
✓ Same prediction task (next admission cancer type)
✓ Classification only (no drug prediction)

LSTM Fold 1/10
Trainable parameters: 3,926,791
Input dimensions: Lab=2560, Rad=2560, DrugDesc=2560, DrugHistory=2374
Epoch  5/50 | Train Loss: 1.3263 | Val Loss: 1.2947
Epoch 10/50 | Train Loss: 0.9247 | Val Loss: 1.6262
Early stopping at epoch 12

Fold 1 Results:
  Micro-F1 : 0.5290
  Macro-F1 : 0.1879
  ROC-AUC  : 0.7358

LSTM Fold 2/10
Trainable parameters: 3,926,791
Input dimensions: Lab=2560, Rad=2560, DrugDesc=2560, DrugHistory=2374
Epoch  5/50 | Train Loss: 1.2981 | Val Loss: 1.3187
Epoch 10/50 | Train Loss: 0.9